In [ ]:
# Install condacolab (helper for conda in Colab)
!pip install -q condacolab
import condacolab
condacolab.install()

In [ ]:
from google.colab import files
files.download('/content/qwag/algeom/weighted_curve.py')

In [ ]:
# Create a conda environment named 'sage' and install SageMath
!conda create -n sage sage python=3.11 -y

In [ ]:
%%bash
source activate sage
sage -c "GF(7)"
sage -c "R.<x> = PolynomialRing(GF(7)); print(x^2 + 1)"

In [ ]:
%%bash
source activate sage
mkdir -p /content/qwag/algeom
touch /content/qwag/__init__.py
touch /content/qwag/algeom/__init__.py
echo "algeom directory created successfully!"

In [ ]:
%%writefile /content/qwag/algeom/weighted_projective_space.py
from sage.all import gcd, PolynomialRing
from itertools import product # Add this import

class WeightedProjectiveSpace:
    def __init__(self, field, weights):
        """
        Initialize weighted projective space P(w_0,...,w_n) over GF(q).
        field: finite field (e.g., GF(7))
        weights: tuple or list of positive integers (e.g., [1,2])
        """
        self.field = field
        self.weights = tuple(weights)
        self.dim = len(weights) - 1

        if not all(w > 0 for w in weights):
            raise ValueError("Weights must be positive integers.")

        if gcd(weights) != 1:
            print("Warning: gcd(weights) != 1 — space may not be well-formed.")

        # Coordinate ring
        var_names = [f'x{i}' for i in range(len(weights))]
        self.coord_ring = PolynomialRing(field, var_names)

    def __repr__(self):
        return f"WeightedProjectiveSpace over {self.field} with weights {self.weights}"

    def is_well_formed(self):
        """Checks if gcd of all weights is 1."""
        from sage.all import gcd
        return gcd(self.weights) == 1

    def rational_points(self):
        """
        Returns a list of all distinct rational points in the weighted projective space
        P(w_0,...,w_n) over self.field.
        Points are returned as tuples of field elements, with a canonical representative chosen
        for each equivalence class. The canonical representative is the lexicographically
        smallest tuple in its class.
        """
        field = self.field
        weights = self.weights
        num_vars = len(weights)
        projective_points = set()

        # Iterate over all possible (n+1)-tuples of field elements
        for p_coords_tuple in product(field, repeat=num_vars):
            # Skip the zero vector (not a projective point)
            if all(c == field(0) for c in p_coords_tuple):
                continue

            # Use field elements directly for reliable hashing and comparison
            # Sage field elements are hashable and comparable.
            canonical_rep = p_coords_tuple

            for lam_val in field.list():
                if lam_val == field(0): # Cannot scale by zero
                    continue

                scaled_p = tuple(lam_val**weights[i] * p_coords_tuple[i] for i in range(num_vars))

                if scaled_p < canonical_rep: # Lexicographical comparison of tuples of field elements
                    canonical_rep = scaled_p

            projective_points.add(canonical_rep)

        return sorted(list(projective_points)) # Sort for consistent output

In [ ]:
%%bash
source activate sage
mkdir -p /root/.local/share/jupyter/kernels
cp -r $SAGE_ROOT/share/jupyter/kernels/sagemath /root/.local/share/jupyter/kernels/ 2>/dev/null || true
echo "Kernel copied — restart session again!"

In [ ]:
%%bash
source activate sage
sage -c "print(GF(7))"

In [ ]:
%%bash
source activate sage
export PYTHONPATH=$PYTHONPATH:/content
sage -c """
from qwag.algeom.weighted_projective_space import WeightedProjectiveSpace

P = WeightedProjectiveSpace(GF(7), [1,2])
print(P)
"""

In [ ]:
%%writefile -a /content/qwag/algeom/weighted_projective_space.py

    def is_well_formed(self):
        """Checks if gcd of all weights is 1."""
        from sage.all import gcd
        return gcd(self.weights) == 1

In [ ]:
%%bash
source activate sage
export PYTHONPATH=$PYTHONPATH:/content
sage -c """
from qwag.algeom.weighted_projective_space import WeightedProjectiveSpace
from sage.all import GF # Explicitly import GF here for the main script context

P = WeightedProjectiveSpace(GF(7), [1,2])
print('Well-formed?', P.is_well_formed())  # Should be True

Q = WeightedProjectiveSpace(GF(7), [2,4])
print('Well-formed?', Q.is_well_formed())  # Should be False (gcd=2)
"""

In [ ]:
%%writefile /content/qwag/algeom/weighted_curve.py
from sage.all import gcd, PolynomialRing
from itertools import product

from .weighted_projective_space import WeightedProjectiveSpace  # Import from same subpackage

class WeightedCurve(WeightedProjectiveSpace):
    def __init__(self, space, polynomial):
        """
        Weighted curve defined by a weighted homogeneous polynomial in the space.
        space: WeightedProjectiveSpace instance
        polynomial: weighted homogeneous polynomial (Sage poly)
        """
        super().__init__(space.field, space.weights)
        self.space = space # Store the space object
        self.poly = polynomial

        # Calculate weighted degree and check for weighted homogeneity
        weighted_degrees_of_terms = []
        if not self.poly: # Handle zero polynomial
            raise ValueError("Polynomial cannot be zero.")

        for exponents_tuple, coeff in self.poly.dict().items():
            term_weighted_degree = sum(exponent * weight for exponent, weight in zip(exponents_tuple, self.weights))
            weighted_degrees_of_terms.append(term_weighted_degree)

        if not weighted_degrees_of_terms:
            raise ValueError("Polynomial is empty (should not happen for non-zero poly).")

        deg = weighted_degrees_of_terms[0]
        if not all(d == deg for d in weighted_degrees_of_terms):
            raise ValueError("Polynomial is not weighted homogeneous with given weights.")

        if deg <= 0:
            raise ValueError("Polynomial must have positive weighted degree.")

        self.weighted_degree = deg # Store the weighted degree

        # Coordinate ring is inherited from space
        self.coord_ring = space.coord_ring

    def __repr__(self):
        return f"WeightedCurve in {self.space} defined by {self.poly}"

    def genus(self):
        """
        Computes the arithmetic genus of the curve.
        This formula is applicable for a smooth hypersurface of weighted degree `d`
        in a weighted projective space of dimension `len(self.weights) - 1`.
        Note: This implementation does not account for singularities.
        """
        d = self.weighted_degree
        ambient_dim = len(self.weights) - 1 # Dimension of the ambient weighted projective space
        g = 1 + (d - 2) * ambient_dim // 2
        return g

    def smooth_rational_points(self):
        """
        Returns smooth (non-singular) rational points on the curve.
        This implementation iterates over affine points and checks for vanishing
        of the polynomial and its partial derivatives.
        Note: This currently returns affine points. Proper projective normalization
        for weighted projective space is more complex and not implemented here.
        """
        field = self.field
        coord_ring = self.coord_ring
        variables = list(coord_ring.gens())
        num_vars = len(self.weights) # n from (x_0, ..., x_{n-1})

        smooth_points = []

        # Calculate partial derivatives
        partial_derivatives = [self.poly.derivative(var) for var in variables]

        # Iterate over all possible affine points (x_0, ..., x_{num_vars-1})
        for p_coords_tuple in product(field, repeat=num_vars):
            # Create a substitution dictionary
            sub_dict = {var: val for var, val in zip(variables, p_coords_tuple)}

            # 1. Check if the point is on the curve (F(P) = 0)
            if self.poly.subs(sub_dict) == 0:
                # 2. Check for smoothness (not all partial derivatives vanish at P)
                if not self.is_singular(p_coords_tuple):
                    smooth_points.append(p_coords_tuple)

        return smooth_points

    def is_singular(self, p):
        """
        Checks if a given point p (tuple of coordinates) on the curve is singular.
        A point is singular if all partial derivatives of the polynomial vanish at that point.
        """
        coord_ring = self.coord_ring
        variables = list(coord_ring.gens())

        # Create a substitution dictionary for the point p
        sub_dict = {var: val for var, val in zip(variables, p)}

        # Calculate partial derivatives (if not already cached)
        partial_derivatives = [self.poly.derivative(var) for var in variables]

        # Check if all partial derivatives vanish at point p
        for derivative_poly in partial_derivatives:
            if derivative_poly.subs(sub_dict) != 0:
                return False  # Not singular, at least one derivative is non-zero
        return True  # All partial derivatives vanish, so it's singular

    def rational_points(self):
        """
        Returns a list of all distinct rational points on the curve.
        It retrieves all rational points from the ambient weighted projective space
        and filters them to include only those that satisfy the curve's equation.
        """
        curve_points = []
        ambient_points = self.space.rational_points() # Get points from the ambient space

        # Get variables from the coordinate ring to create substitution dictionaries
        variables = list(self.coord_ring.gens())

        for p_coords_tuple in ambient_points:
            # Create a substitution dictionary for the point p
            sub_dict = {var: val for var, val in zip(variables, p_coords_tuple)}

            # Check if the point satisfies the curve's polynomial (F(P) = 0)
            if self.poly.subs(sub_dict) == 0:
                curve_points.append(p_coords_tuple)

        return sorted(curve_points)

In [ ]:
%%bash
source activate sage
export PYTHONPATH=$PYTHONPATH:/content
sage -c """
from qwag.algeom.weighted_projective_space import WeightedProjectiveSpace
from qwag.algeom.weighted_curve import WeightedCurve
from sage.all import GF

field = GF(7)
space = WeightedProjectiveSpace(field, [1, 1]) # Changed weights to [1, 1]
x, y = space.coord_ring.gens()
poly = x**3 + y**3  # Changed polynomial to x^3 + y^3 (weighted homogeneous of degree 3)

curve = WeightedCurve(space, poly)
print(curve)
print('Arithmetic Genus:', curve.arithmetic_genus())
print('Singular Rational Points:', curve.singular_rational_points())
print('Geometric Genus (approx):', curve.geometric_genus())
"""

In [ ]:
%%writefile -a /content/qwag/algeom/weighted_curve.py

    def smooth_rational_points(self):
        """
        Returns smooth (non-singular) rational points on the curve.
        Placeholder implementation: returns all rational points for now.
        Full version will check Jacobian vanishing.
        """
        # TODO: Implement singularity check using partial derivatives
        # For now, placeholder: all points (excluding origin if needed)
        # Later: filter with self.is_singular(p)
        return []  # Replace with actual list

In [ ]:
%%bash
source activate sage
export PYTHONPATH=$PYTHONPATH:/content
sage -c """
from qwag.algeom.weighted_projective_space import WeightedProjectiveSpace
from qwag.algeom.weighted_curve import WeightedCurve
from sage.all import GF

field = GF(7)
space = WeightedProjectiveSpace(field, [1, 2])
x, y = space.coord_ring.gens()
poly = y**2 - x**4

curve = WeightedCurve(space, poly)
print(curve)
print('Smooth points:', curve.smooth_rational_points())
"""

In [ ]:
%%bash
cat /content/qwag/algeom/weighted_curve.py | grep '^def ' | sed 's/def //g' | sed 's/(.*//g' | sort -u

In [ ]:
%%bash
cat /content/qwag/algeom/weighted_curve.py | grep -E '^class (CustomLinearCode|WeightedCurve):' -A 1000 | grep -E '^    def ' | sed 's/    def //g' | sed 's/(.*//g' | sort -u

In [ ]:
%%bash
source activate sage
export PYTHONPATH=$PYTHONPATH:/content
sage -c """
from qwag.algeom.weighted_projective_space import WeightedProjectiveSpace
from qwag.algeom.weighted_curve import WeightedCurve
from sage.all import GF

field = GF(9)
weights = [1, 1]
space = WeightedProjectiveSpace(field, weights)
x, y = space.coord_ring.gens()

# Example curve: x^2 + y^2 = 0 over GF(9)
poly = x**2 + y**2
curve = WeightedCurve(space, poly)

print(f'Rational points for {curve}:')
print(curve.rational_points())

field = GF(7)
weights_diff = [1, 2]
space_diff = WeightedProjectiveSpace(field, weights_diff)
x, y = space_diff.coord_ring.gens()

# Example curve: y^2 - x^4 = 0 over GF(7)
poly_diff = y**2 - x**4
curve_diff = WeightedCurve(space_diff, poly_diff)

print(f'Rational points for {curve_diff}:')
print(curve_diff.rational_points())
"""

In [ ]:
%%bash
source activate sage
export PYTHONPATH=$PYTHONPATH:/content
sage -c """
from qwag.algeom.weighted_projective_space import WeightedProjectiveSpace
from qwag.algeom.weighted_curve import WeightedCurve
from sage.all import GF

field = GF(7) # You can change this finite field
weights = [1, 2, 3] # Define your desired weights [q_0, q_1, q_2]
space = WeightedProjectiveSpace(field, weights)

x, y, z = space.coord_ring.gens() # Get the variables x, y, z

# Define a weighted homogeneous polynomial in x, y, z
# Example: x^6 (w_x=1, deg=6), y^3 (w_y=2, deg=6), z^2 (w_z=3, deg=6)
# Ensure all terms have the same weighted degree for weighted homogeneity
poly = x**6 + y**3 + z**2

curve = WeightedCurve(space, poly)

print(curve)
print('Arithmetic Genus:', curve.arithmetic_genus())
print('Singular Rational Points:', curve.singular_rational_points())
print('Geometric Genus (approx):', curve.geometric_genus())
"""

In [ ]:
%%bash
source activate sage
export PYTHONPATH=$PYTHONPATH:/content
sage -c """
from qwag.algeom.weighted_projective_space import WeightedProjectiveSpace
from qwag.algeom.weighted_curve import WeightedCurve
from sage.all import GF

field = GF(11) # A different finite field
weights = [1, 1] # Standard projective plane weights
space = WeightedProjectiveSpace(field, weights)
x, y = space.coord_ring.gens()

# A weighted homogeneous polynomial of degree 3 for weights [1,1]
# x^3 has weighted degree 1*3 = 3
# y^3 has weighted degree 1*3 = 3
poly = x**3 + y**3

curve = WeightedCurve(space, poly)
print(curve)
print('Arithmetic Genus:', curve.arithmetic_genus())
print('Singular Rational Points:', curve.singular_rational_points())
print('Geometric Genus (approx):', curve.geometric_genus())
"""

# Task
Okay, I will refactor the `WeightedCurve` class in `/content/qwag/algeom/weighted_curve.py` by:

1.  Renaming the `genus` method to `arithmetic_genus` and updating its docstring.
2.  Adding a new method `singular_rational_points` to identify singular rational points using `is_singular`.
3.  Adding a new method `geometric_genus` to approximate the geometric genus based on the arithmetic genus and the count of singular rational points, including a disclaimer about its limitations.

After updating the class, I will modify test cells `3pE8t8x85S93`, `29a508be`, and `4af43b5c` to demonstrate these new and updated genus calculations. Finally, I will provide a summary of the implemented genus formulas and their distinctions.

## Rename Arithmetic Genus Method

### Subtask:
Rename the existing `genus` method in `weighted_curve.py` to `arithmetic_genus` to clearly distinguish it from the geometric genus. Update its docstring to reflect this.


**Reasoning**:
I need to read the content of the `weighted_curve.py` file, modify the `genus` method to `arithmetic_genus` and update its docstring, and then write the entire modified content back to the file.



In [ ]:
%%bash
script_content="""
from sage.all import gcd, PolynomialRing
from itertools import product

from .weighted_projective_space import WeightedProjectiveSpace  # Import from same subpackage

class WeightedCurve(WeightedProjectiveSpace):
    def __init__(self, space, polynomial):
        """
        Weighted curve defined by a weighted homogeneous polynomial in the space.
        space: WeightedProjectiveSpace instance
        polynomial: weighted homogeneous polynomial (Sage poly)
        """
        super().__init__(space.field, space.weights)
        self.space = space # Store the space object
        self.poly = polynomial

        # Calculate weighted degree and check for weighted homogeneity
        weighted_degrees_of_terms = []
        if not self.poly: # Handle zero polynomial
            raise ValueError("Polynomial cannot be zero.")

        for exponents_tuple, coeff in self.poly.dict().items():
            term_weighted_degree = sum(exponent * weight for exponent, weight in zip(exponents_tuple, self.weights))
            weighted_degrees_of_terms.append(term_weighted_degree)

        if not weighted_degrees_of_terms:
            raise ValueError("Polynomial is empty (should not happen for non-zero poly).")

        deg = weighted_degrees_of_terms[0]
        if not all(d == deg for d in weighted_degrees_of_terms):
            raise ValueError("Polynomial is not weighted homogeneous with given weights.")

        if deg <= 0:
            raise ValueError("Polynomial must have positive weighted degree.")

        self.weighted_degree = deg # Store the weighted degree

        # Coordinate ring is inherited from space
        self.coord_ring = space.coord_ring

    def __repr__(self):
        return f"WeightedCurve in {self.space} defined by {self.poly}"

    def arithmetic_genus(self):
        """
        Computes the arithmetic genus of the curve.
        This formula is applicable for a smooth hypersurface of weighted degree `d`
        in a weighted projective space of dimension `len(self.weights) - 1`.
        Note: This implementation does not account for singularities and provides the arithmetic genus.
        """
        d = self.weighted_degree
        ambient_dim = len(self.weights) - 1 # Dimension of the ambient weighted projective space
        g = 1 + (d - 2) * ambient_dim // 2
        return g

    def smooth_rational_points(self):
        """
        Returns smooth (non-singular) rational points on the curve.
        This implementation iterates over affine points and checks for vanishing
        of the polynomial and its partial derivatives.
        Note: This currently returns affine points. Proper projective normalization
        for weighted projective space is more complex and not implemented here.
        """
        field = self.field
        coord_ring = self.coord_ring
        variables = list(coord_ring.gens())
        num_vars = len(self.weights) # n from (x_0, ..., x_{n-1})

        smooth_points = []

        # Calculate partial derivatives
        partial_derivatives = [self.poly.derivative(var) for var in variables]

        # Iterate over all possible affine points (x_0, ..., x_{num_vars-1})
        for p_coords_tuple in product(field, repeat=num_vars):
            # Create a substitution dictionary
            sub_dict = {var: val for var, val in zip(variables, p_coords_tuple)}

            # 1. Check if the point is on the curve (F(P) = 0)
            if self.poly.subs(sub_dict) == 0:
                # 2. Check for smoothness (not all partial derivatives vanish at P)
                if not self.is_singular(p_coords_tuple):
                    smooth_points.append(p_coords_tuple)

        return smooth_points

    def is_singular(self, p):
        """
        Checks if a given point p (tuple of coordinates) on the curve is singular.
        A point is singular if all partial derivatives of the polynomial vanish at that point.
        """
        coord_ring = self.coord_ring
        variables = list(coord_ring.gens())

        # Create a substitution dictionary for the point p
        sub_dict = {var: val for var, val in zip(variables, p)}

        # Calculate partial derivatives (if not already cached)
        partial_derivatives = [self.poly.derivative(var) for var in variables]

        # Check if all partial derivatives vanish at point p
        for derivative_poly in partial_derivatives:
            if derivative_poly.subs(sub_dict) != 0:
                return False  # Not singular, at least one derivative is non-zero
        return True  # All partial derivatives vanish, so it's singular

    def rational_points(self):
        """
        Returns a list of all distinct rational points on the curve.
        It retrieves all rational points from the ambient weighted projective space
        and filters them to include only those that satisfy the curve's equation.
        """
        curve_points = []
        ambient_points = self.space.rational_points() # Get points from the ambient space

        # Get variables from the coordinate ring to create substitution dictionaries
        variables = list(self.coord_ring.gens())

        for p_coords_tuple in ambient_points:
            # Create a substitution dictionary for the point p
            sub_dict = {var: val for var, val in zip(variables, p_coords_tuple)}

            # Check if the point satisfies the curve's polynomial (F(P) = 0)
            if self.poly.subs(sub_dict) == 0:
                curve_points.append(p_coords_tuple)

        return sorted(curve_points)
"""

# Use 'echo' to write the content to the file, overwriting it
echo "$script_content" > /content/qwag/algeom/weighted_curve.py

# Verification step (optional, but good practice)
cat /content/qwag/algeom/weighted_curve.py | grep "def arithmetic_genus"

**Reasoning**:
The previous `%%bash` command failed because the shell misinterpreted the multi-line Python string assigned to `script_content` as shell commands due to improper handling of quotes and newlines. I will use `cat << EOF` to robustly write the entire Python script content, which includes renaming the `genus` method to `arithmetic_genus` and updating its docstring, to the specified file.



In [ ]:
%%bash
cat << EOF > /content/qwag/algeom/weighted_curve.py
from sage.all import gcd, PolynomialRing
from itertools import product

from .weighted_projective_space import WeightedProjectiveSpace  # Import from same subpackage

class WeightedCurve(WeightedProjectiveSpace):
    def __init__(self, space, polynomial):
        """
        Weighted curve defined by a weighted homogeneous polynomial in the space.
        space: WeightedProjectiveSpace instance
        polynomial: weighted homogeneous polynomial (Sage poly)
        """
        super().__init__(space.field, space.weights)
        self.space = space # Store the space object
        self.poly = polynomial

        # Calculate weighted degree and check for weighted homogeneity
        weighted_degrees_of_terms = []
        if not self.poly: # Handle zero polynomial
            raise ValueError("Polynomial cannot be zero.")

        for exponents_tuple, coeff in self.poly.dict().items():
            term_weighted_degree = sum(exponent * weight for exponent, weight in zip(exponents_tuple, self.weights))
            weighted_degrees_of_terms.append(term_weighted_degree)

        if not weighted_degrees_of_terms:
            raise ValueError("Polynomial is empty (should not happen for non-zero poly).")

        deg = weighted_degrees_of_terms[0]
        if not all(d == deg for d in weighted_degrees_of_terms):
            raise ValueError("Polynomial is not weighted homogeneous with given weights.")

        if deg <= 0:
            raise ValueError("Polynomial must have positive weighted degree.")

        self.weighted_degree = deg # Store the weighted degree

        # Coordinate ring is inherited from space
        self.coord_ring = space.coord_ring

    def __repr__(self):
        return f"WeightedCurve in {self.space} defined by {self.poly}"

    def arithmetic_genus(self):
        """
        Computes the arithmetic genus of the curve.
        This formula is applicable for a smooth hypersurface of weighted degree `d`
        in a weighted projective space of dimension `len(self.weights) - 1`.
        Note: This implementation does not account for singularities and provides the arithmetic genus.
        """
        d = self.weighted_degree
        ambient_dim = len(self.weights) - 1 # Dimension of the ambient weighted projective space
        g = 1 + (d - 2) * ambient_dim // 2
        return g

    def smooth_rational_points(self):
        """
        Returns smooth (non-singular) rational points on the curve.
        This implementation iterates over affine points and checks for vanishing
        of the polynomial and its partial derivatives.
        Note: This currently returns affine points. Proper projective normalization
        for weighted projective space is more complex and not implemented here.
        """
        field = self.field
        coord_ring = self.coord_ring
        variables = list(coord_ring.gens())
        num_vars = len(self.weights) # n from (x_0, ..., x_{n-1})

        smooth_points = []

        # Calculate partial derivatives
        partial_derivatives = [self.poly.derivative(var) for var in variables]

        # Iterate over all possible affine points (x_0, ..., x_{num_vars-1})
        for p_coords_tuple in product(field, repeat=num_vars):
            # Create a substitution dictionary
            sub_dict = {var: val for var, val in zip(variables, p_coords_tuple)}

            # 1. Check if the point is on the curve (F(P) = 0)
            if self.poly.subs(sub_dict) == 0:
                # 2. Check for smoothness (not all partial derivatives vanish at P)
                if not self.is_singular(p_coords_tuple):
                    smooth_points.append(p_coords_tuple)

        return smooth_points

    def is_singular(self, p):
        """
        Checks if a given point p (tuple of coordinates) on the curve is singular.
        A point is singular if all partial derivatives of the polynomial vanish at that point.
        """
        coord_ring = self.coord_ring
        variables = list(coord_ring.gens())

        # Create a substitution dictionary for the point p
        sub_dict = {var: val for var, val in zip(variables, p)}

        # Calculate partial derivatives (if not already cached)
        partial_derivatives = [self.poly.derivative(var) for var in variables]

        # Check if all partial derivatives vanish at point p
        for derivative_poly in partial_derivatives:
            if derivative_poly.subs(sub_dict) != 0:
                return False  # Not singular, at least one derivative is non-zero
        return True  # All partial derivatives vanish, so it's singular

    def rational_points(self):
        """
        Returns a list of all distinct rational points on the curve.
        It retrieves all rational points from the ambient weighted projective space
        and filters them to include only those that satisfy the curve's equation.
        """
        curve_points = []
        ambient_points = self.space.rational_points() # Get points from the ambient space

        # Get variables from the coordinate ring to create substitution dictionaries
        variables = list(self.coord_ring.gens())

        for p_coords_tuple in ambient_points:
            # Create a substitution dictionary for the point p
            sub_dict = {var: val for var, val in zip(variables, p_coords_tuple)}

            # Check if the point satisfies the curve's polynomial (F(P) = 0)
            if self.poly.subs(sub_dict) == 0:
                curve_points.append(p_coords_tuple)

        return sorted(curve_points)
EOF

# Verification step
cat /content/qwag/algeom/weighted_curve.py | grep "def arithmetic_genus"

**Reasoning**:
The previous `cat << EOF` command failed because bash was interpreting the Python f-string syntax (e.g., `{self.space}`) as shell variables. To prevent bash from parsing the content of the heredoc, I need to quote the `EOF` marker as `'EOF'`. This ensures the Python code is written literally to the file.



In [ ]:
%%bash
cat << 'EOF' > /content/qwag/algeom/weighted_curve.py
from sage.all import gcd, PolynomialRing
from itertools import product

from .weighted_projective_space import WeightedProjectiveSpace  # Import from same subpackage

class WeightedCurve(WeightedProjectiveSpace):
    def __init__(self, space, polynomial):
        """
        Weighted curve defined by a weighted homogeneous polynomial in the space.
        space: WeightedProjectiveSpace instance
        polynomial: weighted homogeneous polynomial (Sage poly)
        """
        super().__init__(space.field, space.weights)
        self.space = space # Store the space object
        self.poly = polynomial

        # Calculate weighted degree and check for weighted homogeneity
        weighted_degrees_of_terms = []
        if not self.poly: # Handle zero polynomial
            raise ValueError("Polynomial cannot be zero.")

        for exponents_tuple, coeff in self.poly.dict().items():
            term_weighted_degree = sum(exponent * weight for exponent, weight in zip(exponents_tuple, self.weights))
            weighted_degrees_of_terms.append(term_weighted_degree)

        if not weighted_degrees_of_terms:
            raise ValueError("Polynomial is empty (should not happen for non-zero poly).")

        deg = weighted_degrees_of_terms[0]
        if not all(d == deg for d in weighted_degrees_of_terms):
            raise ValueError("Polynomial is not weighted homogeneous with given weights.")

        if deg <= 0:
            raise ValueError("Polynomial must have positive weighted degree.")

        self.weighted_degree = deg # Store the weighted degree

        # Coordinate ring is inherited from space
        self.coord_ring = space.coord_ring

    def __repr__(self):
        return f"WeightedCurve in {self.space} defined by {self.poly}"

    def arithmetic_genus(self):
        """
        Computes the arithmetic genus of the curve.
        This formula is applicable for a smooth hypersurface of weighted degree `d`
        in a weighted projective space of dimension `len(self.weights) - 1`.
        Note: This implementation does not account for singularities and provides the arithmetic genus.
        """
        d = self.weighted_degree
        ambient_dim = len(self.weights) - 1 # Dimension of the ambient weighted projective space
        g = 1 + (d - 2) * ambient_dim // 2
        return g

    def smooth_rational_points(self):
        """
        Returns smooth (non-singular) rational points on the curve.
        This implementation iterates over affine points and checks for vanishing
        of the polynomial and its partial derivatives.
        Note: This currently returns affine points. Proper projective normalization
        for weighted projective space is more complex and not implemented here.
        """
        field = self.field
        coord_ring = self.coord_ring
        variables = list(coord_ring.gens())
        num_vars = len(self.weights) # n from (x_0, ..., x_{n-1})

        smooth_points = []

        # Calculate partial derivatives
        partial_derivatives = [self.poly.derivative(var) for var in variables]

        # Iterate over all possible affine points (x_0, ..., x_{num_vars-1})
        for p_coords_tuple in product(field, repeat=num_vars):
            # Create a substitution dictionary
            sub_dict = {var: val for var, val in zip(variables, p_coords_tuple)}

            # 1. Check if the point is on the curve (F(P) = 0)
            if self.poly.subs(sub_dict) == 0:
                # 2. Check for smoothness (not all partial derivatives vanish at P)
                if not self.is_singular(p_coords_tuple):
                    smooth_points.append(p_coords_tuple)

        return smooth_points

    def is_singular(self, p):
        """
        Checks if a given point p (tuple of coordinates) on the curve is singular.
        A point is singular if all partial derivatives of the polynomial vanish at that point.
        """
        coord_ring = self.coord_ring
        variables = list(coord_ring.gens())

        # Create a substitution dictionary for the point p
        sub_dict = {var: val for var, val in zip(variables, p)}

        # Calculate partial derivatives (if not already cached)
        partial_derivatives = [self.poly.derivative(var) for var in variables]

        # Check if all partial derivatives vanish at point p
        for derivative_poly in partial_derivatives:
            if derivative_poly.subs(sub_dict) != 0:
                return False  # Not singular, at least one derivative is non-zero
        return True  # All partial derivatives vanish, so it's singular

    def rational_points(self):
        """
        Returns a list of all distinct rational points on the curve.
        It retrieves all rational points from the ambient weighted projective space
        and filters them to include only those that satisfy the curve's equation.
        """
        curve_points = []
        ambient_points = self.space.rational_points() # Get points from the ambient space

        # Get variables from the coordinate ring to create substitution dictionaries
        variables = list(self.coord_ring.gens())

        for p_coords_tuple in ambient_points:
            # Create a substitution dictionary for the point p
            sub_dict = {var: val for var, val in zip(variables, p_coords_tuple)}

            # Check if the point satisfies the curve's polynomial (F(P) = 0)
            if self.poly.subs(sub_dict) == 0:
                curve_points.append(p_coords_tuple)

        return sorted(curve_points)
EOF

# Verification step
cat /content/qwag/algeom/weighted_curve.py | grep "def arithmetic_genus"


**Reasoning**:
The previous step successfully updated the `weighted_curve.py` file by renaming the `genus` method to `arithmetic_genus` and updating its docstring. Now, as per the main task, I will add a new method `singular_rational_points` to the `WeightedCurve` class, which will identify singular rational points by filtering `rational_points` using the `is_singular` method.



In [ ]:
%%writefile -a /content/qwag/algeom/weighted_curve.py

    def singular_rational_points(self):
        """
        Returns a list of singular rational points on the curve.
        These are points on the curve where all partial derivatives of the defining
        polynomial vanish simultaneously.
        """
        singular_points = []
        all_rational_points = self.rational_points() # Get all points on the curve

        for p_coords_tuple in all_rational_points:
            if self.is_singular(p_coords_tuple):
                singular_points.append(p_coords_tuple)
        return sorted(singular_points)


**Reasoning**:
As per the main task, I will now add a new method `geometric_genus` to the `WeightedCurve` class. This method will approximate the geometric genus based on the arithmetic genus and the count of singular rational points, and will include a disclaimer about its limitations.



In [ ]:
%%writefile -a /content/qwag/algeom/weighted_curve.py

    def geometric_genus(self):
        """
        Approximates the geometric genus (p_g) of the curve.
        For a curve with ordinary singularities, the geometric genus can be estimated
        as p_g = p_a - sum(delta_i) where p_a is the arithmetic genus and delta_i
        are contributions from each singularity. For this approximation, we will
        simply subtract the number of singular rational points from the arithmetic genus.

        Disclaimer: This is a highly simplified approximation and does not account
        for the complex nature and multiplicity of singularities. It serves as a
        basic lower bound for the geometric genus for non-singular cases, and
        a rough estimation for singular cases. For precise geometric genus, more
        advanced techniques are required.
        """
        # A very simplified approximation: geometric genus <= arithmetic genus
        # and roughly p_g = p_a - number_of_singular_points
        # This assumes each singular point reduces the genus by 1, which is not generally true
        # for all types of singularities or for non-ordinary singularities.
        num_singular_points = len(self.singular_rational_points())
        approx_geometric_genus = self.arithmetic_genus() - num_singular_points
        return max(0, approx_geometric_genus) # Genus cannot be negative

**Reasoning**:
The previous steps successfully modified the `WeightedCurve` class. Now, I need to update cell `3pE8t8x85S93` to use the newly renamed `arithmetic_genus` method and display the result of the new `geometric_genus` method, as well as the singular rational points.



In [ ]:
%%bash
source activate sage
export PYTHONPATH=$PYTHONPATH:/content
sage -c """
from qwag.algeom.weighted_projective_space import WeightedProjectiveSpace
from qwag.algeom.weighted_curve import WeightedCurve
from sage.all import GF

field = GF(7)
space = WeightedProjectiveSpace(field, [1, 2])
x, y = space.coord_ring.gens()
poly = y**2 - x**4  # Weighted homogeneous of degree 4

curve = WeightedCurve(space, poly)
print(curve)
print('Arithmetic Genus:', curve.arithmetic_genus())
print('Singular Rational Points:', curve.singular_rational_points())
print('Geometric Genus (approx):', curve.geometric_genus())
"""

**Reasoning**:
I need to update cell `29a508be` to use the new `arithmetic_genus`, `singular_rational_points`, and `geometric_genus` methods. This involves modifying the existing code to call these new methods and print their results.



In [ ]:
%%bash
source activate sage
export PYTHONPATH=$PYTHONPATH:/content
sage -c """
from qwag.algeom.weighted_projective_space import WeightedProjectiveSpace
from qwag.algeom.weighted_curve import WeightedCurve
from sage.all import GF

field = GF(7)
weights = [1, 2, 3]
space = WeightedProjectiveSpace(field, weights)

x, y, z = space.coord_ring.gens()

poly = x**6 + y**3 + z**2

curve = WeightedCurve(space, poly)

print(curve)
print('Arithmetic Genus:', curve.arithmetic_genus())
print('Singular Rational Points:', curve.singular_rational_points())
print('Geometric Genus (approx):', curve.geometric_genus())
"""

**Reasoning**:
I need to update cell `4af43b5c` to use the new `arithmetic_genus`, `singular_rational_points`, and `geometric_genus` methods. This involves modifying the existing code to call these new methods and print their results.



In [ ]:
%%bash
source activate sage
export PYTHONPATH=$PYTHONPATH:/content
sage -c """
from qwag.algeom.weighted_projective_space import WeightedProjectiveSpace
from qwag.algeom.weighted_curve import WeightedCurve
from sage.all import GF

field = GF(11) # A different finite field
weights = [1, 1] # Standard projective plane weights
space = WeightedProjectiveSpace(field, weights)
x, y = space.coord_ring.gens()

# A weighted homogeneous polynomial of degree 3 for weights [1,1]
# x^3 has weighted degree 1*3 = 3
# y^3 has weighted degree 1*3 = 3
poly = x**3 + y**3

curve = WeightedCurve(space, poly)
print(curve)
print('Arithmetic Genus:', curve.arithmetic_genus())
print('Singular Rational Points:', curve.singular_rational_points())
print('Geometric Genus (approx):', curve.geometric_genus())
"""

In [ ]:
%%writefile /content/qwag/algeom/weighted_curve.py
from sage.all import gcd, PolynomialRing
from itertools import product

from .weighted_projective_space import WeightedProjectiveSpace  # Import from same subpackage

# Define the LinearCode class at the module level
class LinearCode:
    def __init__(self, generator_matrix):
        self.generator_matrix = generator_matrix
        self.n = generator_matrix.ncols()
        self.k = generator_matrix.rank() # Correctly calculate k as the rank of the generator matrix

    def __repr__(self):
        return f"LinearCode with parameters [n={self.n}, k={self.k}]\nGenerator matrix:\n{self.generator_matrix}"

    def is_self_orthogonal(self):
        """
        Checks if the linear code is self-orthogonal, i.e., C subset C^perp.
        This means that every codeword is orthogonal to every other codeword (including itself).
        For a generator matrix G, this is equivalent to G * G^T == 0.
        """
        # We need the generator matrix in its original form, not necessarily echelon form
        # However, the echelon form has the same row space, so it works for orthogonality check.
        if self.generator_matrix.is_zero():
            return True

        # G * G.transpose() must be the zero matrix
        result_matrix = self.generator_matrix * self.generator_matrix.transpose()
        return result_matrix.is_zero()


# Helper function for generating weighted monomials (free function in the module)
def _generate_weighted_monomials(poly_ring, weights, max_weighted_degree):
    """
    Generates a list of monomials in `poly_ring` whose weighted degree is less than or equal to
    `max_weighted_degree`.
    """
    monomials = []
    num_vars = len(weights)
    variables = list(poly_ring.gens())

    if max_weighted_degree < 0:
        return []

    def __generate_recursive_exponents(current_exponents_list, current_weighted_degree, var_idx):
        if var_idx == num_vars:
            monomial = poly_ring(1)
            for i in range(num_vars):
                monomial *= variables[i]**current_exponents_list[i]
            monomials.append(monomial)
            return

        if weights[var_idx] == 0:
            max_exponent_for_var = 0
        else:
            max_exponent_for_var = (max_weighted_degree - current_weighted_degree) // weights[var_idx]

        for exponent in range(max_exponent_for_var + 1):
            __generate_recursive_exponents(
                current_exponents_list + [exponent],
                current_weighted_degree + exponent * weights[var_idx],
                var_idx + 1
            )

    __generate_recursive_exponents([], 0, 0)
    return monomials


class WeightedCurve(WeightedProjectiveSpace):
    def __init__(self, space, polynomial):
        """
        Weighted curve defined by a weighted homogeneous polynomial in the space.
        space: WeightedProjectiveSpace instance
        polynomial: weighted homogeneous polynomial (Sage poly)
        """
        super().__init__(space.field, space.weights)
        self.space = space # Store the space object
        self.poly = polynomial

        # Calculate weighted degree and check for weighted homogeneity
        weighted_degrees_of_terms = []
        if not self.poly: # Handle zero polynomial
            raise ValueError("Polynomial cannot be zero.")

        for exponents_tuple, coeff in self.poly.dict().items():
            term_weighted_degree = sum(exponent * weight for exponent, weight in zip(exponents_tuple, self.weights))
            weighted_degrees_of_terms.append(term_weighted_degree)

        if not weighted_degrees_of_terms:
            raise ValueError("Polynomial is empty (should not happen for non-zero poly).")

        deg = weighted_degrees_of_terms[0]
        if not all(d == deg for d in weighted_degrees_of_terms):
            raise ValueError("Polynomial is not weighted homogeneous with given weights.")

        if deg <= 0:
            raise ValueError("Polynomial must have positive weighted degree.")

        self.weighted_degree = deg # Store the weighted degree

        # Coordinate ring is inherited from space
        self.coord_ring = space.coord_ring

    def __repr__(self):
        return f"WeightedCurve in {self.space} defined by {self.poly}"

    def genus(self):
        """
        Computes the arithmetic genus of the curve.
        This formula is applicable for a smooth hypersurface of weighted degree `d`
        in a weighted projective space of dimension `len(self.weights) - 1`.
        Note: This implementation does not account for singularities.
        """
        d = self.weighted_degree
        ambient_dim = len(self.weights) - 1 # Dimension of the ambient weighted projective space
        g = 1 + (d - 2) * ambient_dim // 2
        return g

    def smooth_rational_points(self):
        """
        Returns smooth (non-singular) rational points on the curve.
        This implementation iterates over affine points and checks for vanishing
        of the polynomial and its partial derivatives.
        Note: This currently returns affine points. Proper projective normalization
        for weighted projective space is more complex and not implemented here.
        """
        field = self.field
        coord_ring = self.coord_ring
        variables = list(coord_ring.gens())
        num_vars = len(self.weights) # n from (x_0, ..., x_{n-1})

        smooth_points = []

        # Calculate partial derivatives
        partial_derivatives = [self.poly.derivative(var) for var in variables]

        # Iterate over all possible affine points (x_0, ..., x_{num_vars-1})
        for p_coords_tuple in product(field, repeat=num_vars):
            # Create a substitution dictionary
            sub_dict = {var: val for var, val in zip(variables, p_coords_tuple)}

            # 1. Check if the point is on the curve (F(P) = 0)
            if self.poly.subs(sub_dict) == 0:
                # 2. Check for smoothness (not all partial derivatives vanish at P)
                if not self.is_singular(p_coords_tuple):
                    smooth_points.append(p_coords_tuple)

        return smooth_points

    def is_singular(self, p):
        """
        Checks if a given point p (tuple of coordinates) on the curve is singular.
        A point is singular if all partial derivatives of the polynomial vanish at that point.
        """
        coord_ring = self.coord_ring
        variables = list(coord_ring.gens())

        # Create a substitution dictionary for the point p
        sub_dict = {var: val for var, val in zip(variables, p)}

        # Calculate partial derivatives (if not already cached)
        partial_derivatives = [self.poly.derivative(var) for var in variables]

        # Check if all partial derivatives vanish at point p
        for derivative_poly in partial_derivatives:
            if derivative_poly.subs(sub_dict) != 0:
                return False  # Not singular, at least one derivative is non-zero
        return True  # All partial derivatives vanish, so it's singular

    def rational_points(self):
        """
        Returns a list of all distinct rational points on the curve.
        It retrieves all rational points from the ambient weighted projective space
        and filters them to include only those that satisfy the curve's equation.
        """
        curve_points = []
        ambient_points = self.space.rational_points() # Get points from the ambient space

        # Get variables from the coordinate ring to create substitution dictionaries
        variables = list(self.coord_ring.gens())

        for p_coords_tuple in ambient_points:
            # Create a substitution dictionary for the point p
            sub_dict = {var: val for var, val in zip(variables, p_coords_tuple)}

            # Check if the point satisfies the curve's polynomial (F(P) = 0)
            if self.poly.subs(sub_dict) == 0:
                curve_points.append(p_coords_tuple)

        return sorted(curve_points)

    def riemann_roch_basis(self, divisor_degree):
        """
        Computes a basis for the Riemann-Roch space L(D) for a divisor D of degree `divisor_degree`.
        This implementation constructs a basis of weighted homogeneous polynomials of
        weighted degree `divisor_degree` that are distinct in the quotient ring K[x0, ..., xn] / <F>,
        where F is the curve's defining polynomial. It uses a Groebner basis to find unique representatives.

        Note: This is a computational interpretation suitable for finding a basis
        for polynomial functions on the curve of a specific weighted degree. For a general
        divisor on a weighted projective curve, a formal basis for L(D) involves rational functions
        and requires more advanced algebraic geometry (e.g., canonical divisors, sheaf cohomology).
        The actual dimension l(D) = max(0, divisor_degree - self.geometric_genus() + 1)
        holds for sufficiently large `divisor_degree`.
        """
        if divisor_degree < 0:
            return []
        if divisor_degree == 0:
            return [self.coord_ring(1)] # Space of constant functions

        # Generate candidate monomials up to the desired weighted degree
        candidate_monomials = _generate_weighted_monomials(
            self.coord_ring, self.weights, divisor_degree
        )

        # Compute a Groebner basis for the ideal generated by the curve polynomial
        ideal = self.coord_ring.ideal(self.poly)
        gb = ideal.groebner_basis()

        # Reduce all candidate monomials modulo the Groebner basis to find unique representatives
        basis_elements_reduced = set()
        for monomial in candidate_monomials:
            reduced_form = monomial.reduce(gb)
            basis_elements_reduced.add(reduced_form)

        # Sort the unique basis elements for consistent output
        return sorted(list(basis_elements_reduced), key=str)

    def evaluation_code(self, points_list, divisor_degree):
        """
        Creates a generator matrix for a linear code based on evaluating
        functions from a Riemann-Roch space at a set of rational points.

        Args:
            points_list (list): A list of rational points (tuples of field elements)
                                 at which to evaluate the basis functions.
            divisor_degree (int): The degree 'G' for which to compute the Riemann-Roch basis.

        Returns:
            LinearCode: An object representing the linear code.
        """
        from sage.all import matrix # Import matrix from sage.all

        basis = self.riemann_roch_basis(divisor_degree)
        gen_matrix = []

        if not basis or not points_list:
            # Handle cases where there's no basis or no points to evaluate at
            return LinearCode(matrix(self.field, 0, 0)) # Return empty code

        for f in basis:
            row = []
            for p in points_list:
                # Create a substitution dictionary for the point p
                variables = list(self.coord_ring.gens())
                sub_dict = {var: val for var, val in zip(variables, p)}
                row.append(f.subs(sub_dict))
            gen_matrix.append(row)

        # If gen_matrix is empty, matrix() will raise an error, handle that
        if not gen_matrix or not gen_matrix[0]: # Check if the matrix would be empty
            return LinearCode(matrix(self.field, 0, 0))

        G_matrix = matrix(self.field, gen_matrix)

        # Return a simplified LinearCode object using the echelon form of the matrix
        return LinearCode(G_matrix.echelon_form())

In [ ]:
%%bash
source activate sage
export PYTHONPATH=$PYTHONPATH:/content
sage -c """
from qwag.algeom.weighted_projective_space import WeightedProjectiveSpace
from qwag.algeom.weighted_curve import WeightedCurve, LinearCode # Import LinearCode
from sage.all import GF

field = GF(7)
space = WeightedProjectiveSpace(field, [1, 2])
x, y = space.coord_ring.gens()
poly = y**2 - x**4  # Weighted homogeneous of degree 4

curve = WeightedCurve(space, poly)

# Get rational points on the curve
rational_points = curve.rational_points()

# Choose a divisor degree
divisor_degree = 2

# Generate the evaluation code
eval_code = curve.evaluation_code(rational_points, divisor_degree)

print(f'Linear Code generated:\n{eval_code}')
print(f'Is the Linear Code self-orthogonal? {eval_code.is_self_orthogonal()}')
"""

In [ ]:
%%bash
source activate sage
export PYTHONPATH=$PYTHONPATH:/content
sage -c """
from qwag.algeom.weighted_projective_space import WeightedProjectiveSpace
from qwag.algeom.weighted_curve import WeightedCurve, LinearCode # Import LinearCode
from sage.all import GF

field = GF(7)
space = WeightedProjectiveSpace(field, [1, 2])
x, y = space.coord_ring.gens()
poly = y**2 - x**4  # Weighted homogeneous of degree 4

curve = WeightedCurve(space, poly)

# Get rational points on the curve
rational_points = curve.rational_points()
print(f'Curve: {curve}')
print(f'Rational points on curve: {rational_points}')

# Choose a divisor degree (e.g., 2)
divisor_degree = 2

# Generate the evaluation code
eval_code = curve.evaluation_code(rational_points, divisor_degree)
print(f'\nEvaluation Code (Divisor Degree {divisor_degree}):')
print(eval_code)
"""

## Final Task

### Subtask:
Confirm that the `riemann_roch_basis` method has been added and remind the user about the missing `generate_weighted_monomials` dependency, asking for its implementation.


## Summary:

### Q&A
*   **Has the `riemann_roch_basis` method been added?**
    Yes, the `riemann_roch_basis` method has been successfully added to the `WeightedCurve` class.
*   **Is the `generate_weighted_monomials` dependency still missing?**
    No, the dependency is no longer missing. A helper function named `_generate_weighted_monomials` has been implemented within the same file and is now utilized by the `riemann_roch_basis` method.

### Data Analysis Key Findings
*   The `riemann_roch_basis` method, along with a new internal helper function `_generate_weighted_monomials`, has been appended to the `/content/qwag/algeom/weighted_curve.py` file.
*   The `riemann_roch_basis` method now computes a basis for a Riemann-Roch-like space by generating candidate monomials up to a specified weighted degree using `_generate_weighted_monomials`.
*   The implementation of `riemann_roch_basis` incorporates Groebner bases to reduce candidate monomials modulo the curve's defining polynomial, thereby finding unique representatives for the basis elements.
*   The `_generate_weighted_monomials` helper function is a recursive function designed to produce all monomials within a given polynomial ring whose weighted degree does not exceed a specified maximum.

### Insights or Next Steps
*   The `riemann_roch_basis` method provides a more robust and algebraically sound computation for a basis of polynomial functions on the curve, leveraging Groebner bases for reduction.
*   The naming convention `_generate_weighted_monomials` suggests its intended use as an internal helper, ensuring modularity within the `weighted_curve.py` file.


# Task
Let's delve into the explanation of self-orthogonal code generation from algebraic-geometric curves.

### Explanation of Self-Orthogonal Code Generation from Algebraic-Geometric Curves

Self-orthogonal codes are linear codes where every codeword is orthogonal to every other codeword, including itself. In the context of algebraic-geometric (AG) codes, this property is intimately linked to the underlying curve, the chosen divisor, and the characteristics of the finite field.

The general construction of an AG code involves:
1.  **A curve `X`** defined over a finite field `F_q`.
2.  **A set of `n` distinct `F_q`-rational points `P_1, ..., P_n`** on the curve `X`. These points define the length `n` of the code.
3.  **A divisor `D = P_1 + ... + P_n`**.
4.  **A divisor `G`** whose support does not intersect the support of `D`. The degree of `G`, denoted `deg(G)`, is crucial for the code's parameters.
5.  **A Riemann-Roch space `L(G)`** consisting of rational functions on `X` whose poles are restricted by `G`. This space forms the message space of the code.

The code `C` is then defined by evaluating the functions in `L(G)` at the points `P_1, ..., P_n`. That is,
`C = { (f(P_1), ..., f(P_n)) | f \in L(G) }`.

The generator matrix `G_C` for such a code is formed by rows corresponding to a basis of `L(G)`, evaluated at the points `P_i`. The dimension of the code `k` is typically `dim(L(G))`, which can be computed using the Riemann-Roch theorem: `k = dim(L(G)) = deg(G) - g + 1` (for `deg(G) > 2g - 2`, where `g` is the genus of the curve).

#### Factors Influencing Self-Orthogonality (`G_C * G_C^T = 0`):

For an AG code to be self-orthogonal, its generator matrix `G_C` must satisfy `G_C * G_C^T = 0`. This condition translates to specific properties related to the chosen curve, divisor, and field.

1.  **Choice of Curve `X`**:
    *   **Genus `g`**: The genus of the curve is a fundamental invariant that affects the dimension of the Riemann-Roch spaces. Curves with certain genera might be more conducive to self-orthogonal constructions.
    *   **Existence of Rational Points**: The number of rational points `n` determines the code length. For self-orthogonality, we typically require `n` to be large enough, and the distribution of these points can be important.
    *   **Symmetry and Automorphism Groups**: Curves with rich automorphism groups (symmetries) can lead to codes with desirable algebraic properties, including self-orthogonality, if the divisor `G` is chosen appropriately with respect to these symmetries.

2.  **Choice of Divisor `G` (Divisor Degree `deg(G)`)**:
    *   **Degree of `G`**: The degree `deg(G)` directly influences the dimension `k` of the code. A key condition for self-orthogonality in many constructions is related to the parity or specific value of `deg(G)` relative to other curve parameters.
    *   **Canonical Divisor**: The canonical divisor `K` (which has degree `2g-2`) plays a special role. Codes are often related to `L(G)` where `G` is chosen such that `2G` is linearly equivalent to `K + D` or some other combination that forces orthogonality.
    *   **Symmetric Divisors**: If `G` has certain symmetries (e.g., if it's invariant under certain automorphisms), it can simplify the self-orthogonality condition.

3.  **Field Characteristic `char(F_q)`**:
    *   **Characteristic 2**: When the characteristic of the field `q` is 2, the self-orthogonality condition simplifies significantly. In characteristic 2, `x^2 = (x + ... + x) = 0` for any `x` in the field if there are an even number of `x`'s. This often makes it easier to construct self-orthogonal codes. Many powerful self-orthogonal codes, like extended Golay codes, exist over `F_2`.
    *   **Other Characteristics**: For fields with `char(F_q) != 2`, the self-orthogonality condition `G_C * G_C^T = 0` is more restrictive. Non-zero elements can have non-zero squares, so careful selection of functions and points is required.

#### General Construction Methods for Self-Orthogonal AG Codes:

1.  **The Adjoint Code `C_L(D, G)^perp` (Dual Code Perspective)**:
    The dual code of an AG code `C_L(D,G)` is given by `C_Omega(D, K-G+D)`, where `K` is a canonical divisor.
    A code `C` is self-orthogonal if `C \subseteq C^perp`. This implies that the functions in `L(G)` must satisfy certain conditions related to the functions in `L(K-G+D)`. This often boils down to:
    *   The divisor `G` should be chosen such that `2G \sim K + D` (linearly equivalent), or a similar condition that makes `L(G)` small enough and "orthogonal" to itself.
    *   In many cases, if `2G` is related to `K` and `D` in a specific way, it can lead to self-orthogonal codes.

2.  **Specific Curve Families**:
    *   **Hermitian Curves**: These curves `y^q + y = x^(q+1)` over `F_(q^2)` are known to yield good self-orthogonal codes, especially in characteristic 2. Their structure (high genus, many rational points) makes them suitable.
    *   **Deligne-Lusztig Curves**: Another class of curves that provides a rich source of AG codes.

3.  **Trace Codes**:
    Codes obtained by applying a trace function from an extension field to a subfield often exhibit self-orthogonality properties, particularly when specific types of curves and divisors are used.

4.  **Quadratic Form Codes**:
    Some algebraic constructions of self-orthogonal codes directly use quadratic forms over finite fields, and these can be linked to properties of curves.

In summary, generating self-orthogonal codes from AG curves is a complex task that requires a deep understanding of algebraic geometry, particularly the theory of Riemann-Roch spaces and divisors. The choice of curve (genus, number of points), the field characteristic (especially characteristic 2 being favorable), and the specific properties of the divisor `G` are all critical. The `is_self_orthogonal` method we implemented (checking `G * G^T = 0`) is the direct computational test for this property, but finding curves and divisors that *guarantee* this property is the challenge in constructive coding theory.

### Next Steps for Implementation:

Based on this explanation, I can offer to:

*   **Explore Concrete Examples**: We could try to implement or analyze a known construction of a self-orthogonal code from a specific type of curve (e.g., a Hermitian curve) to see how the parameters `D` and `G` lead to the `G * G^T = 0` condition. This would involve selecting a field characteristic (e.g., GF(4) or GF(8)), defining a specific curve, choosing points and a divisor `G`, and then evaluating the self-orthogonality of the resulting code.
*   **Search for Relevant Literature**: I can search for academic papers or textbooks that provide explicit constructions of self-orthogonal AG codes and attempt to reproduce one of them in our framework. This would likely involve more advanced concepts like differential forms or more sophisticated Riemann-Roch calculations.

Please let me know which direction you'd prefer to take!

## Explain Self-Orthogonal Code Generation

### Subtask:
Provide a detailed explanation of the factors influencing self-orthogonal code generation from algebraic-geometric curves, including choice of curve, divisor degree, field characteristic, and general construction methods.


## Explanation of Self-Orthogonal Code Generation from Algebraic-Geometric Curves

Self-orthogonal codes are a special class of linear codes where every codeword is orthogonal to every other codeword (including itself). This property is crucial in coding theory for various applications, including constructing self-dual codes and understanding the structure of codes.

### General Construction of an Algebraic-Geometric (AG) Code:

AG codes are constructed from algebraic curves defined over finite fields. The general steps involve:

1.  **Choice of Curve (X):** Select a smooth, projective, geometrically irreducible algebraic curve $X$ defined over a finite field $\mathbb{F}_q$.
2.  **Rational Points (P):** Identify a set of $n$ distinct $\mathbb{F}_q$-rational points $P = \{P_1, \ldots, P_n\}$ on $X$.
3.  **Divisors (D and G):**
    *   **$D = P_1 + \ldots + P_n$**: The sum of the $n$ distinct rational points. This defines the "evaluation points" for the code.
    *   **$G$**: A divisor on $X$ whose support is disjoint from $D$. The degree of $G$, denoted $\deg(G)$, is a key parameter.
4.  **Riemann-Roch Space ($L(G)$):** Construct the $\mathbb{F}_q$-vector space $L(G) = \{f \in \mathbb{F}_q(X)^* \mid \div(f) + G \ge 0\} \cup \{0\}$, where $\mathbb{F}_q(X)$ is the function field of $X$. This space consists of functions whose poles are constrained by $G$.
5.  **Evaluation Map:** Define an evaluation map $ev_D: L(G) \to \mathbb{F}_q^n$ by $ev_D(f) = (f(P_1), \ldots, f(P_n))$.
6.  **Code (C):** The AG code $C$ is the image of $L(G)$ under the evaluation map, $C = ev_D(L(G))$. The generator matrix of $C$ is formed by taking a basis of $L(G)$ and evaluating each basis function at the points in $D$. The dimension $k$ of the code is given by $\dim_{\mathbb{F}_q} L(G)$. The length $n$ is the number of rational points chosen.

### Factors Influencing Self-Orthogonality:

The self-orthogonality of an AG code ($C \subseteq C^{\perp}$) is strongly influenced by several factors:

1.  **Choice of Curve ($X$):**
    *   **Genus ($g$):** The genus of the curve plays a fundamental role in the dimension of $L(G)$ via the Riemann-Roch Theorem. Curves with lower genus tend to be simpler, but higher genus curves can yield more complex codes. For self-orthogonal codes, specific curve geometries or symmetries are often exploited.
    *   **Number and Distribution of Rational Points:** The number of $\mathbb{F}_q$-rational points dictates the code length $n$. The distribution of these points can influence the properties of the evaluation map and thus the code.
    *   **Symmetry and Automorphism Groups:** Curves with large automorphism groups (e.g., Hermitian curves) often provide natural symmetries that can be used to construct self-orthogonal codes by selecting points and divisors appropriately.

2.  **Choice of Divisor $G$ (Degree, Canonical Divisor, Symmetric Divisors):**
    *   **Degree of $G$ ($\deg(G)$):** This is critical. For a code $C = C(D, G)$ to be self-orthogonal, the degree of $G$ must satisfy certain conditions, often related to the code's designed distance. Specifically, for self-orthogonality, it's frequently required that $2 \deg(G) < n$ (the number of points). The condition $2G \sim K$ (where $K$ is the canonical divisor) or similar constructions can be used to achieve self-orthogonality, though this is related to self-duality.
    *   **Canonical Divisor ($K$):** The canonical divisor, related to the differential forms on the curve, plays a central role in the duality theorem for AG codes. If $2G$ is linearly equivalent to $K$, the code can be self-dual or self-orthogonal under certain conditions.
    *   **Symmetric Divisors:** Selecting divisors $G$ that exhibit certain symmetries with respect to the curve or the set of evaluation points $D$ can lead to self-orthogonal codes. For example, if $G$ is chosen such that $2G$ is symmetric or related to $K$ and $D$.

3.  **Field Characteristic:**
    *   **Characteristic 2 vs. Others:** The characteristic of the finite field $\mathbb{F}_q$ is highly significant. In characteristic 2, the definition of orthogonality (dot product) changes, and self-orthogonal codes are sometimes easier to construct or exhibit different properties compared to fields of odd characteristic. For instance, many classical self-dual codes exist only in characteristic 2. The property $f(P) \cdot f(P) = 0$ is naturally satisfied in characteristic 2 for certain constructions.

### General Construction Methods for Self-Orthogonal AG Codes:

1.  **Adjoint Codes (Dual Codes):** A fundamental approach involves constructing the dual code $C^{\perp}$. Self-orthogonal codes are those where $C \subseteq C^{\perp}$. The dual of an AG code $C(D, G)$ is typically an AG code $C(D, D-G+K)$, where $K$ is the canonical divisor. Conditions for self-orthogonality can then be derived by forcing the image of $L(G)$ to be contained in the image of $L(D-G+K)$.

2.  **Specific Curve Families:**
    *   **Hermitian Curves:** These curves (e.g., $y^q + y = x^{q+1}$ over $\mathbb{F}_{q^2}$) are known for having many rational points and large automorphism groups. They are a rich source for constructing self-orthogonal and self-dual codes, especially in characteristic 2.
    *   **Deligne-Lusztig Curves:** These are another class of curves with interesting properties that can be leveraged for code construction.

3.  **Trace Codes:** Codes can be constructed by taking a basis of functions over an extension field and then applying the trace map down to the base field. This method can sometimes yield self-orthogonal codes, particularly in the context of Goppa codes.

4.  **Quadratic Form Codes:** Codes related to quadratic forms can be constructed on certain algebraic curves, particularly in even characteristic. The orthogonality condition naturally arises from the properties of the quadratic form. The geometric interpretation of the ambient space and the curve's embedding can be crucial.

5.  **Using Automorphisms:** If the curve has an automorphism $\sigma$ with certain properties, and $G$ is a $\sigma$-invariant divisor, it's often possible to construct codes that are self-orthogonal with respect to a $\sigma$-invariant inner product.

In essence, generating self-orthogonal AG codes involves a careful interplay between the geometry of the curve (genus, points, symmetries), the algebraic properties of the chosen divisors (degree, relation to canonical divisor), and the arithmetic of the base finite field (especially its characteristic).

## Explore concrete example: Self-orthogonal code over GF(2)

### Subtask:
Demonstrate the generation of a self-orthogonal code using `WeightedCurve`, `riemann_roch_basis`, `evaluation_code`, and `is_self_orthogonal` with parameters chosen to yield a self-orthogonal code.


**Reasoning**:
I need to demonstrate the generation of a self-orthogonal code by performing a series of steps: importing necessary classes, defining a finite field and weighted projective space, creating a weighted curve with a specific polynomial, retrieving rational points, setting a divisor degree, generating an evaluation code, and finally printing the code and checking its self-orthogonality. All these steps can be consolidated into a single code block to be executed by SageMath via a bash command.



In [ ]:
%%bash
source activate sage
export PYTHONPATH=$PYTHONPATH:/content
sage -c """
from sage.all import GF
from qwag.algeom.weighted_projective_space import WeightedProjectiveSpace
from qwag.algeom.weighted_curve import WeightedCurve, LinearCode

# 1. Define the finite field as GF(2).
field = GF(2)

# 2. Define a weighted projective space using the GF(2) field and weights [1, 1].
weights = [1, 1]
space = WeightedProjectiveSpace(field, weights)

# 3. Get the variables (x, y) from the coordinate ring of the defined space.
x, y = space.coord_ring.gens()

# 4. Define a weighted homogeneous polynomial poly = x*y.
poly = x*y

# 5. Instantiate the WeightedCurve class with the created space and the polynomial.
curve = WeightedCurve(space, poly)

# 6. Retrieve the rational points on this curve using the curve.rational_points() method.
rational_points = curve.rational_points()
print(f'Curve: {curve}')
print(f'Rational points on curve: {rational_points}')

# 7. Set the divisor_degree to 0.
divisor_degree = 0

# 8. Generate the evaluation code by calling curve.evaluation_code() with the rational_points and divisor_degree.
eval_code = curve.evaluation_code(rational_points, divisor_degree)

# 9. Print the generated LinearCode object.
print(f'\nEvaluation Code (Divisor Degree {divisor_degree}):')
print(eval_code)

# 10. Print the result of checking for self-orthogonality.
print(f'Is the Linear Code self-orthogonal? {eval_code.is_self_orthogonal()}')
"""

# Task
Add a `dual_code` method to the `LinearCode` class in `/content/qwag/algeom/weighted_curve.py`. This method should return the dual code (`C^perp`) of the given linear code, leveraging SageMath's linear code functionality.

## Add Dual Code Method to LinearCode

### Subtask:
Add a `dual_code` method to the `LinearCode` class in `/content/qwag/algeom/weighted_curve.py`. This method will return the dual code (`C^perp`) of the given linear code.


**Reasoning**:
I need to add a `dual_code` method to the `LinearCode` class in `/content/qwag/algeom/weighted_curve.py`. This method will create a SageMath `LinearCode` object, compute its dual, and then return a new instance of our custom `LinearCode` class with the dual code's generator matrix. I will use `cat << 'EOF'` to ensure the multi-line Python code is written correctly to the file, and then verify the addition of the new method.



In [ ]:
%%bash
cat << 'EOF' > /content/qwag/algeom/weighted_curve.py
from sage.all import gcd, PolynomialRing
from itertools import product

from .weighted_projective_space import WeightedProjectiveSpace  # Import from same subpackage

# Define the LinearCode class at the module level
class LinearCode:
    def __init__(self, generator_matrix):
        self.generator_matrix = generator_matrix
        self.n = generator_matrix.ncols()
        self.k = generator_matrix.rank() # Correctly calculate k as the rank of the generator matrix

    def __repr__(self):
        return f"LinearCode with parameters [n={self.n}, k={self.k}]\nGenerator matrix:\n{self.generator_matrix}"

    def is_self_orthogonal(self):
        """
        Checks if the linear code is self-orthogonal, i.e., C subset C^perp.
        This means that every codeword is orthogonal to every other codeword (including itself).
        For a generator matrix G, this is equivalent to G * G^T == 0.
        """
        # We need the generator matrix in its original form, not necessarily echelon form
        # However, the echelon form has the same row space, so it works for orthogonality check.
        if self.generator_matrix.is_zero():
            return True

        # G * G.transpose() must be the zero matrix
        result_matrix = self.generator_matrix * self.generator_matrix.transpose()
        return result_matrix.is_zero()

    def dual_code(self):
        """
        Returns the dual code (C^perp) of the current linear code.
        The dual code is defined as the set of all vectors that are orthogonal to every codeword in the original code.
        """
        from sage.codes.linear_code import LinearCode as SageLinearCode # Import to avoid circular dependency

        # Create a SageMath LinearCode instance from our generator matrix
        sage_code = SageLinearCode(self.generator_matrix)

        # Compute the dual code using SageMath's functionality
        sage_dual_code = sage_code.dual_code()

        # Return a new instance of our custom LinearCode class with the dual code's generator matrix
        return LinearCode(sage_dual_code.generator_matrix())

# Helper function for generating weighted monomials (free function in the module)
def _generate_weighted_monomials(poly_ring, weights, max_weighted_degree):
    """
    Generates a list of monomials in `poly_ring` whose weighted degree is less than or equal to
    `max_weighted_degree`.
    """
    monomials = []
    num_vars = len(weights)
    variables = list(poly_ring.gens())

    if max_weighted_degree < 0:
        return []

    def __generate_recursive_exponents(current_exponents_list, current_weighted_degree, var_idx):
        if var_idx == num_vars:
            monomial = poly_ring(1)
            for i in range(num_vars):
                monomial *= variables[i]**current_exponents_list[i]
            monomials.append(monomial)
            return

        if weights[var_idx] == 0:
            max_exponent_for_var = 0
        else:
            max_exponent_for_var = (max_weighted_degree - current_weighted_degree) // weights[var_idx]

        for exponent in range(max_exponent_for_var + 1):
            __generate_recursive_exponents(
                current_exponents_list + [exponent],
                current_weighted_degree + exponent * weights[var_idx],
                var_idx + 1
            )

    __generate_recursive_exponents([], 0, 0)
    return monomials


class WeightedCurve(WeightedProjectiveSpace):
    def __init__(self, space, polynomial):
        """
        Weighted curve defined by a weighted homogeneous polynomial in the space.
        space: WeightedProjectiveSpace instance
        polynomial: weighted homogeneous polynomial (Sage poly)
        """
        super().__init__(space.field, space.weights)
        self.space = space # Store the space object
        self.poly = polynomial

        # Calculate weighted degree and check for weighted homogeneity
        weighted_degrees_of_terms = []
        if not self.poly: # Handle zero polynomial
            raise ValueError("Polynomial cannot be zero.")

        for exponents_tuple, coeff in self.poly.dict().items():
            term_weighted_degree = sum(exponent * weight for exponent, weight in zip(exponents_tuple, self.weights))
            weighted_degrees_of_terms.append(term_weighted_degree)

        if not weighted_degrees_of_terms:
            raise ValueError("Polynomial is empty (should not happen for non-zero poly).")

        deg = weighted_degrees_of_terms[0]
        if not all(d == deg for d in weighted_degrees_of_terms):
            raise ValueError("Polynomial is not weighted homogeneous with given weights.")

        if deg <= 0:
            raise ValueError("Polynomial must have positive weighted degree.")

        self.weighted_degree = deg # Store the weighted degree

        # Coordinate ring is inherited from space
        self.coord_ring = space.coord_ring

    def __repr__(self):
        return f"WeightedCurve in {self.space} defined by {self.poly}"

    def genus(self):
        """
        Computes the arithmetic genus of the curve.
        This formula is applicable for a smooth hypersurface of weighted degree `d`
        in a weighted projective space of dimension `len(self.weights) - 1`.
        Note: This implementation does not account for singularities.
        """
        d = self.weighted_degree
        ambient_dim = len(self.weights) - 1 # Dimension of the ambient weighted projective space
        g = 1 + (d - 2) * ambient_dim // 2
        return g

    def smooth_rational_points(self):
        """
        Returns smooth (non-singular) rational points on the curve.
        This implementation iterates over affine points and checks for vanishing
        of the polynomial and its partial derivatives.
        Note: This currently returns affine points. Proper projective normalization
        for weighted projective space is more complex and not implemented here.
        """
        field = self.field
        coord_ring = self.coord_ring
        variables = list(coord_ring.gens())
        num_vars = len(self.weights) # n from (x_0, ..., x_{n-1})

        smooth_points = []

        # Calculate partial derivatives
        partial_derivatives = [self.poly.derivative(var) for var in variables]

        # Iterate over all possible affine points (x_0, ..., x_{num_vars-1})
        for p_coords_tuple in product(field, repeat=num_vars):
            # Create a substitution dictionary
            sub_dict = {var: val for var, val in zip(variables, p_coords_tuple)}

            # 1. Check if the point is on the curve (F(P) = 0)
            if self.poly.subs(sub_dict) == 0:
                # 2. Check for smoothness (not all partial derivatives vanish at P)
                if not self.is_singular(p_coords_tuple):
                    smooth_points.append(p_coords_tuple)

        return smooth_points

    def is_singular(self, p):
        """
        Checks if a given point p (tuple of coordinates) on the curve is singular.
        A point is singular if all partial derivatives of the polynomial vanish at that point.
        """
        coord_ring = self.coord_ring
        variables = list(coord_ring.gens())

        # Create a substitution dictionary for the point p
        sub_dict = {var: val for var, val in zip(variables, p)}

        # Calculate partial derivatives (if not already cached)
        partial_derivatives = [self.poly.derivative(var) for var in variables]

        # Check if all partial derivatives vanish at point p
        for derivative_poly in partial_derivatives:
            if derivative_poly.subs(sub_dict) != 0:
                return False  # Not singular, at least one derivative is non-zero
        return True  # All partial derivatives vanish, so it's singular

    def rational_points(self):
        """
        Returns a list of all distinct rational points on the curve.
        It retrieves all rational points from the ambient weighted projective space
        and filters them to include only those that satisfy the curve's equation.
        """
        curve_points = []
        ambient_points = self.space.rational_points() # Get points from the ambient space

        # Get variables from the coordinate ring to create substitution dictionaries
        variables = list(self.coord_ring.gens())

        for p_coords_tuple in ambient_points:
            # Create a substitution dictionary for the point p
            sub_dict = {var: val for var, val in zip(variables, p_coords_tuple)}

            # Check if the point satisfies the curve's polynomial (F(P) = 0)
            if self.poly.subs(sub_dict) == 0:
                curve_points.append(p_coords_tuple)

        return sorted(curve_points)

    def riemann_roch_basis(self, divisor_degree):
        """
        Computes a basis for the Riemann-Roch space L(D) for a divisor D of degree `divisor_degree`.
        This implementation constructs a basis of weighted homogeneous polynomials of
        weighted degree `divisor_degree` that are distinct in the quotient ring K[x0, ..., xn] / <F>,
        where F is the curve's defining polynomial. It uses a Groebner basis to find unique representatives.

        Note: This is a computational interpretation suitable for finding a basis
        for polynomial functions on the curve of a specific weighted degree. For a general
        divisor on a weighted projective curve, a formal basis for L(D) involves rational functions
        and requires more advanced algebraic geometry (e.g., canonical divisors, sheaf cohomology).
        The actual dimension l(D) = max(0, divisor_degree - self.geometric_genus() + 1)
        holds for sufficiently large `divisor_degree`.
        """
        if divisor_degree < 0:
            return []
        if divisor_degree == 0:
            return [self.coord_ring(1)] # Space of constant functions

        # Generate candidate monomials up to the desired weighted degree
        candidate_monomials = _generate_weighted_monomials(
            self.coord_ring, self.weights, divisor_degree
        )

        # Compute a Groebner basis for the ideal generated by the curve polynomial
        ideal = self.coord_ring.ideal(self.poly)
        gb = ideal.groebner_basis()

        # Reduce all candidate monomials modulo the Groebner basis to find unique representatives
        basis_elements_reduced = set()
        for monomial in candidate_monomials:
            reduced_form = monomial.reduce(gb)
            basis_elements_reduced.add(reduced_form)

        # Sort the unique basis elements for consistent output
        return sorted(list(basis_elements_reduced), key=str)

    def evaluation_code(self, points_list, divisor_degree):
        """
        Creates a generator matrix for a linear code based on evaluating
        functions from a Riemann-Roch space at a set of rational points.

        Args:
            points_list (list): A list of rational points (tuples of field elements)
                                 at which to evaluate the basis functions.
            divisor_degree (int): The degree 'G' for which to compute the Riemann-Roch basis.

        Returns:
            LinearCode: An object representing the linear code.
        """
        from sage.all import matrix # Import matrix from sage.all

        basis = self.riemann_roch_basis(divisor_degree)
        gen_matrix = []

        if not basis or not points_list:
            # Handle cases where there's no basis or no points to evaluate at
            return LinearCode(matrix(self.field, 0, 0)) # Return empty code

        for f in basis:
            row = []
            for p in points_list:
                # Create a substitution dictionary for the point p
                variables = list(self.coord_ring.gens())
                sub_dict = {var: val for var, val in zip(variables, p)}
                row.append(f.subs(sub_dict))
            gen_matrix.append(row)

        # If gen_matrix is empty, matrix() will raise an error, handle that
        if not gen_matrix or not gen_matrix[0]: # Check if the matrix would be empty
            return LinearCode(matrix(self.field, 0, 0))

        G_matrix = matrix(self.field, gen_matrix)

        # Return a simplified LinearCode object using the echelon form of the matrix
        return LinearCode(G_matrix.echelon_form())
EOF

# Verification step (optional, but good practice)
cat /content/qwag/algeom/weighted_curve.py | grep "def dual_code"

## Define Construct CSS Quantum Code Function

### Subtask:
Define a new function, `construct_css_quantum_code`, that takes a `LinearCode` object (which is assumed to be self-orthogonal, i.e., `C ⊆ C^perp`). It will use the `dual_code` method to derive `C_X` and `C_Z` (typically `C_X = C^perp` and `C_Z = C`), calculate the quantum code parameters `n_Q` (length) and `k_Q` (logical qubits), and include a placeholder for `d_Q` (minimum distance) as its calculation is complex and depends on additional classical code properties not yet implemented.


**Reasoning**:
The user wants to add a new function `construct_css_quantum_code` to the `weighted_curve.py` file. This function will take a `LinearCode` object and calculate quantum code parameters. I will append this function to the file using `%%writefile -a` and then verify its addition.



In [ ]:
%%writefile -a /content/qwag/algeom/weighted_curve.py

def construct_css_quantum_code(classical_code):
    """
    Constructs a CSS (Calderbank-Shor-Steane) quantum code from a self-orthogonal classical linear code.

    Args:
        classical_code (LinearCode): A classical linear code object (instance of our LinearCode class)
                                     that is assumed to be self-orthogonal (C \u2286 C^perp).

    Returns:
        dict: A dictionary containing the quantum code parameters (n_Q, k_Q, d_Q).
    """
    # 1. Assert that the classical_code is self-orthogonal
    assert classical_code.is_self_orthogonal(), "The classical code must be self-orthogonal for CSS construction."

    # 2. Define C_Z as the input classical_code and C_X as its dual
    C_Z = classical_code
    C_X = classical_code.dual_code()

    # 3. Calculate quantum code parameters
    n_Q = C_Z.n  # Quantum code length is the classical code length

    # k_Q = dim(C_X) - dim(C_Z) = k_X - k_Z
    # For a self-orthogonal code C \u2286 C^perp, C_Z = C and C_X = C^perp.
    # The number of logical qubits k_Q = k(C_X) - k(C_Z)
    k_Q = C_X.k - C_Z.k

    # d_Q (minimum distance) calculation is complex and depends on additional classical code properties
    # not yet implemented. Set as None for now.
    d_Q = None

    return {
        'n_Q': n_Q,
        'k_Q': k_Q,
        'd_Q': d_Q,
        'C_X': C_X, # Include for completeness/debugging
        'C_Z': C_Z  # Include for completeness/debugging
    }


**Reasoning**:
The user explicitly asked to verify the function has been added by searching for `def construct_css_quantum_code` in the file. I will use a bash command to `grep` for this string in the `weighted_curve.py` file.



In [ ]:
%%bash
cat /content/qwag/algeom/weighted_curve.py | grep "def construct_css_quantum_code"

**Reasoning**:
Now that the `construct_css_quantum_code` function has been added and verified, I will demonstrate its usage by creating a self-orthogonal code and then applying the function to calculate the quantum code parameters.



In [ ]:
%%bash
source activate sage
export PYTHONPATH=$PYTHONPATH:/content
sage -c """
from sage.all import GF
from qwag.algeom.weighted_projective_space import WeightedProjectiveSpace
from qwag.algeom.weighted_curve import WeightedCurve, LinearCode, construct_css_quantum_code

# 1. Define the finite field as GF(2).
field = GF(2)

# 2. Define a weighted projective space using the GF(2) field and weights [1, 1].
weights = [1, 1]
space = WeightedProjectiveSpace(field, weights)

# 3. Get the variables (x, y) from the coordinate ring of the defined space.
x, y = space.coord_ring.gens()

# 4. Define a weighted homogeneous polynomial poly = x*y.
poly = x*y

# 5. Instantiate the WeightedCurve class with the created space and the polynomial.
curve = WeightedCurve(space, poly)

# 6. Retrieve the rational points on this curve using the curve.rational_points() method.
rational_points = curve.rational_points()

# 7. Set the divisor_degree to 0.
divisor_degree = 0

# 8. Generate the evaluation code by calling curve.evaluation_code() with the rational_points and divisor_degree.
eval_code = curve.evaluation_code(rational_points, divisor_degree)

# 9. Verify self-orthogonality (should be True for this example)
if eval_code.is_self_orthogonal():
    print(f'Classical code is self-orthogonal: {eval_code.is_self_orthogonal()}')
    # 10. Construct the CSS quantum code
    quantum_code_params = construct_css_quantum_code(eval_code)

    # 11. Print the quantum code parameters
    print('\nConstructed CSS Quantum Code Parameters:')
    print(f'  n_Q (code length): {quantum_code_params["n_Q"]}')
    print(f'  k_Q (logical qubits): {quantum_code_params["k_Q"]}')
    print(f'  d_Q (minimum distance): {quantum_code_params["d_Q"]}')
    print(f'  C_X generator matrix:\n{quantum_code_params["C_X"].generator_matrix}')
    print(f'  C_Z generator matrix:\n{quantum_code_params["C_Z"].generator_matrix}')
else:
    print(f'Classical code is NOT self-orthogonal: {eval_code.is_self_orthogonal()}')

"""

**Reasoning**:
The previous attempt to import `sage.codes.linear_code` failed because of a `ModuleNotFoundError`. To resolve this, I will import `LinearCode` directly from `sage.all` at the module level in `/content/qwag/algeom/weighted_curve.py` and rename our custom `LinearCode` class to `CustomLinearCode` to avoid name collisions. This change will require re-writing the entire file. Afterward, I will re-run the demonstration code with the updated class name.



In [ ]:
%%writefile /content/qwag/algeom/weighted_curve.py
from sage.all import gcd, PolynomialRing
from itertools import product
from sage.all import LinearCode # Import SageMath's LinearCode

from .weighted_projective_space import WeightedProjectiveSpace  # Corrected import

# Define our custom LinearCode class
class CustomLinearCode:
    def __init__(self, generator_matrix):
        self.generator_matrix = generator_matrix
        self.n = generator_matrix.ncols()
        self.k = generator_matrix.rank() # Correctly calculate k as the rank of the generator matrix

    def __repr__(self):
        return f"CustomLinearCode with parameters [n={self.n}, k={self.k}]\nGenerator matrix:\n{self.generator_matrix}"

    def is_self_orthogonal(self):
        """
        Checks if the linear code is self-orthogonal, i.e., C subset C^perp.
        This means that every codeword is orthogonal to every other codeword (including itself).
        For a generator matrix G, this is equivalent to G * G^T == 0.
        """
        if self.generator_matrix.is_zero():
            return True

        result_matrix = self.generator_matrix * self.generator_matrix.transpose()
        return result_matrix.is_zero()

    def dual_code(self):
        """
        Returns the dual code (C^perp) of the current linear code.
        The dual code is defined as the set of all vectors that are orthogonal to every codeword in the original code.
        """
        # Create a SageMath LinearCode instance using the globally imported Sage LinearCode
        sage_code = LinearCode(self.generator_matrix)

        # Compute the dual code using SageMath's functionality
        sage_dual_code = sage_code.dual_code()

        # Return a new instance of our custom CustomLinearCode class with the dual code's generator matrix
        return CustomLinearCode(sage_dual_code.generator_matrix())

# Helper function for generating weighted monomials (free function in the module)
def _generate_weighted_monomials(poly_ring, weights, max_weighted_degree):
    """
    Generates a list of monomials in `poly_ring` whose weighted degree is less than or equal to
    `max_weighted_degree`.
    """
    monomials = []
    num_vars = len(weights)
    variables = list(poly_ring.gens())

    if max_weighted_degree < 0:
        return []

    def __generate_recursive_exponents(current_exponents_list, current_weighted_degree, var_idx):
        if var_idx == num_vars:
            monomial = poly_ring(1)
            for i in range(num_vars):
                monomial *= variables[i]**current_exponents_list[i]
            monomials.append(monomial)
            return

        if weights[var_idx] == 0:
            max_exponent_for_var = 0
        else:
            max_exponent_for_var = (max_weighted_degree - current_weighted_degree) // weights[var_idx]

        for exponent in range(max_exponent_for_var + 1):
            __generate_recursive_exponents(
                current_exponents_list + [exponent],
                current_weighted_degree + exponent * weights[var_idx],
                var_idx + 1
            )

    __generate_recursive_exponents([], 0, 0)
    return monomials


class WeightedCurve(WeightedProjectiveSpace):
    def __init__(self, space, polynomial):
        """
        Weighted curve defined by a weighted homogeneous polynomial in the space.
        space: WeightedProjectiveSpace instance
        polynomial: weighted homogeneous polynomial (Sage poly)
        """
        super().__init__(space.field, space.weights)
        self.space = space # Store the space object
        self.poly = polynomial

        # Calculate weighted degree and check for weighted homogeneity
        weighted_degrees_of_terms = []
        if not self.poly: # Handle zero polynomial
            raise ValueError("Polynomial cannot be zero.")

        for exponents_tuple, coeff in self.poly.dict().items():
            term_weighted_degree = sum(exponent * weight for exponent, weight in zip(exponents_tuple, self.weights))
            weighted_degrees_of_terms.append(term_weighted_degree)

        if not weighted_degrees_of_terms:
            raise ValueError("Polynomial is empty (should not happen for non-zero poly).")

        deg = weighted_degrees_of_terms[0]
        if not all(d == deg for d in weighted_degrees_of_terms):
            raise ValueError("Polynomial is not weighted homogeneous with given weights.")

        if deg <= 0:
            raise ValueError("Polynomial must have positive weighted degree.")

        self.weighted_degree = deg # Store the weighted degree

        # Coordinate ring is inherited from space
        self.coord_ring = space.coord_ring

    def __repr__(self):
        return f"WeightedCurve in {self.space} defined by {self.poly}"

    def arithmetic_genus(self):
        """
        Computes the arithmetic genus of the curve.
        This formula is applicable for a smooth hypersurface of weighted degree `d`
        in a weighted projective space of dimension `len(self.weights) - 1`.
        Note: This implementation does not account for singularities and provides the arithmetic genus.
        """
        d = self.weighted_degree
        ambient_dim = len(self.weights) - 1 # Dimension of the ambient weighted projective space
        g = 1 + (d - 2) * ambient_dim // 2
        return g

    def smooth_rational_points(self):
        """
        Returns smooth (non-singular) rational points on the curve.
        This implementation iterates over affine points and checks for vanishing
        of the polynomial and its partial derivatives.
        Note: This currently returns affine points. Proper projective normalization
        for weighted projective space is more complex and not implemented here.
        """
        field = self.field
        coord_ring = self.coord_ring
        variables = list(coord_ring.gens())
        num_vars = len(self.weights) # n from (x_0, ..., x_{n-1})

        smooth_points = []

        # Calculate partial derivatives
        partial_derivatives = [self.poly.derivative(var) for var in variables]

        # Iterate over all possible affine points (x_0, ..., x_{num_vars-1})
        for p_coords_tuple in product(field, repeat=num_vars):
            # Create a substitution dictionary
            sub_dict = {var: val for var, val in zip(variables, p_coords_tuple)}

            # 1. Check if the point is on the curve (F(P) = 0)
            if self.poly.subs(sub_dict) == 0:
                # 2. Check for smoothness (not all partial derivatives vanish at P)
                if not self.is_singular(p_coords_tuple):
                    smooth_points.append(p_coords_tuple)

        return smooth_points

    def is_singular(self, p):
        """
        Checks if a given point p (tuple of coordinates) on the curve is singular.
        A point is singular if all partial derivatives of the polynomial vanish at that point.
        """
        coord_ring = self.coord_ring
        variables = list(coord_ring.gens())

        # Create a substitution dictionary for the point p
        sub_dict = {var: val for var, val in zip(variables, p)}

        # Calculate partial derivatives (if not already cached)
        partial_derivatives = [self.poly.derivative(var) for var in variables]

        # Check if all partial derivatives vanish at point p
        for derivative_poly in partial_derivatives:
            if derivative_poly.subs(sub_dict) != 0:
                return False  # Not singular, at least one derivative is non-zero
        return True  # All partial derivatives vanish, so it's singular

    def rational_points(self):
        """
        Returns a list of all distinct rational points on the curve.
        It retrieves all rational points from the ambient weighted projective space
        and filters them to include only those that satisfy the curve's equation.
        """
        curve_points = []
        ambient_points = self.space.rational_points() # Get points from the ambient space

        # Get variables from the coordinate ring to create substitution dictionaries
        variables = list(self.coord_ring.gens())

        for p_coords_tuple in ambient_points:
            # Create a substitution dictionary for the point p
            sub_dict = {var: val for var, val in zip(variables, p_coords_tuple)}

            # Check if the point satisfies the curve's polynomial (F(P) = 0)
            if self.poly.subs(sub_dict) == 0:
                curve_points.append(p_coords_tuple)

        return sorted(curve_points)

    def singular_rational_points(self):
        """
        Returns a list of singular rational points on the curve.
        These are points on the curve where all partial derivatives of the defining
        polynomial vanish simultaneously.
        """
        singular_points = []
        all_rational_points = self.rational_points() # Get all points on the curve

        for p_coords_tuple in all_rational_points:
            if self.is_singular(p_coords_tuple):
                singular_points.append(p_coords_tuple)
        return sorted(singular_points)

    def geometric_genus(self):
        """
        Approximates the geometric genus (p_g) of the curve.
        For a curve with ordinary singularities, the geometric genus can be estimated
        as p_g = p_a - sum(delta_i) where p_a is the arithmetic genus and delta_i
        are contributions from each singularity. For this approximation, we will
        simply subtract the number of singular rational points from the arithmetic genus.

        Disclaimer: This is a highly simplified approximation and does not account
        for the complex nature and multiplicity of singularities. It serves as a
        basic lower bound for the geometric genus for non-singular cases, and
        a rough estimation for singular cases. For precise geometric genus, more
        advanced techniques are required.
        """
        # A very simplified approximation: geometric genus <= arithmetic genus
        # and roughly p_g = p_a - number_of_singular_points
        # This assumes each singular point reduces the genus by 1, which is not generally true
        # for all types of singularities or for non-ordinary singularities.
        num_singular_points = len(self.singular_rational_points())
        approx_geometric_genus = self.arithmetic_genus() - num_singular_points
        return max(0, approx_geometric_genus) # Genus cannot be negative

    def riemann_roch_basis(self, divisor_degree):
        """
        Computes a basis for the Riemann-Roch space L(D) for a divisor D of degree `divisor_degree`.
        """
        if divisor_degree < 0:
            return []
        if divisor_degree == 0:
            return [self.coord_ring(1)] # Space of constant functions

        # Generate candidate monomials up to the desired weighted degree
        candidate_monomials = _generate_weighted_monomials(
            self.coord_ring, self.weights, divisor_degree
        )

        # Compute a Groebner basis for the ideal generated by the curve polynomial
        ideal = self.coord_ring.ideal(self.poly)
        gb = ideal.groebner_basis()

        # Reduce all candidate monomials modulo the Groebner basis to find unique representatives
        basis_elements_reduced = set()
        for monomial in candidate_monomials:
            reduced_form = monomial.reduce(gb)
            basis_elements_reduced.add(reduced_form)

        # Sort the unique basis elements for consistent output
        return sorted(list(basis_elements_reduced), key=str)

    def evaluation_code(self, points_list, divisor_degree):
        """
        Creates a generator matrix for a linear code based on evaluating
        functions from a Riemann-Roch space at a set of rational points.
        """
        from sage.all import matrix # Import matrix from sage.all

        basis = self.riemann_roch_basis(divisor_degree)
        gen_matrix = []

        if not basis or not points_list:
            return CustomLinearCode(matrix(self.field, 0, 0)) # Return empty CustomLinearCode

        for f in basis:
            row = []
            for p in points_list:
                variables = list(self.coord_ring.gens())
                sub_dict = {var: val for var, val in zip(variables, p)}
                row.append(f.subs(sub_dict))
            gen_matrix.append(row)

        if not gen_matrix or not gen_matrix[0]:
            return CustomLinearCode(matrix(self.field, 0, 0))

        G_matrix = matrix(self.field, gen_matrix)

        # Return a CustomLinearCode object
        return CustomLinearCode(G_matrix.echelon_form())

# Function to construct CSS quantum code
def construct_css_quantum_code(classical_code):
    """
    Constructs a CSS (Calderbank-Shor-Steane) quantum code from a self-orthogonal classical linear code.
    """
    assert classical_code.is_self_orthogonal(), "The classical code must be self-orthogonal for CSS construction."

    C_Z = classical_code
    C_X = classical_code.dual_code()

    n_Q = C_Z.n
    k_Q = C_X.k - C_Z.k
    d_Q = None

    return {
        'n_Q': n_Q,
        'k_Q': k_Q,
        'd_Q': d_Q,
        'C_X': C_X,
        'C_Z': C_Z
    }

**Reasoning**:
The `weighted_curve.py` file has been updated to use `CustomLinearCode` and import SageMath's `LinearCode` correctly. Now I will re-run the demonstration code, making sure to import `CustomLinearCode` and use it throughout the script to verify the fix and demonstrate the `construct_css_quantum_code` function.



In [ ]:
%%bash
source activate sage
export PYTHONPATH=$PYTHONPATH:/content
sage -c """
from sage.all import GF
from qwag.algeom.weighted_projective_space import WeightedProjectiveSpace
from qwag.algeom.weighted_curve import WeightedCurve, CustomLinearCode, construct_css_quantum_code

# 1. Define the finite field as GF(2).
field = GF(2)

# 2. Define a weighted projective space using the GF(2) field and weights [1, 1].
weights = [1, 1]
space = WeightedProjectiveSpace(field, weights)

# 3. Get the variables (x, y) from the coordinate ring of the defined space.
x, y = space.coord_ring.gens()

# 4. Define a weighted homogeneous polynomial poly = x*y.
poly = x*y

# 5. Instantiate the WeightedCurve class with the created space and the polynomial.
curve = WeightedCurve(space, poly)

# 6. Retrieve the rational points on this curve using the curve.rational_points() method.
rational_points = curve.rational_points()

# 7. Set the divisor_degree to 0.
divisor_degree = 0

# 8. Generate the evaluation code by calling curve.evaluation_code() with the rational_points and divisor_degree.
eval_code = curve.evaluation_code(rational_points, divisor_degree)

# 9. Verify self-orthogonality (should be True for this example)
if eval_code.is_self_orthogonal():
    print(f'Classical code is self-orthogonal: {eval_code.is_self_orthogonal()}')
    # 10. Construct the CSS quantum code
    quantum_code_params = construct_css_quantum_code(eval_code)

    # Extract parameters into local variables for robust printing
    n_Q_val = quantum_code_params['n_Q']
    k_Q_val = quantum_code_params['k_Q']
    d_Q_val = quantum_code_params['d_Q']
    C_X_gen_matrix = quantum_code_params['C_X'].generator_matrix
    C_Z_gen_matrix = quantum_code_params['C_Z'].generator_matrix

    # 11. Print the quantum code parameters
    print('\nConstructed CSS Quantum Code Parameters:')
    print(f'  n_Q (code length): {n_Q_val}')
    print(f'  k_Q (logical qubits): {k_Q_val}')
    print(f'  d_Q (minimum distance): {d_Q_val}')
    print(f'  C_X generator matrix:\n{C_X_gen_matrix}')
    print(f'  C_Z generator matrix:\n{C_Z_gen_matrix}')
else:
    print(f'Classical code is NOT self-orthogonal: {eval_code.is_self_orthogonal()}')

"""

**Reasoning**:
The `NameError: name 'n_Q' is not defined` indicates a failure to properly access the `n_Q` key from the `quantum_code_params` dictionary within the f-string during execution in the `sage -c` environment. To resolve this, I will explicitly extract the values from the `quantum_code_params` dictionary into local variables before printing them, which is a robust workaround for such issues.



In [ ]:
%%bash
source activate sage
export PYTHONPATH=$PYTHONPATH:/content
sage -c """
from sage.all import GF
from qwag.algeom.weighted_projective_space import WeightedProjectiveSpace
from qwag.algeom.weighted_curve import WeightedCurve, CustomLinearCode, construct_css_quantum_code

# 1. Define the finite field as GF(2).
field = GF(2)

# 2. Define a weighted projective space using the GF(2) field and weights [1, 1].
weights = [1, 1]
space = WeightedProjectiveSpace(field, weights)

# 3. Get the variables (x, y) from the coordinate ring of the defined space.
x, y = space.coord_ring.gens()

# 4. Define a weighted homogeneous polynomial poly = x*y.
poly = x*y

# 5. Instantiate the WeightedCurve class with the created space and the polynomial.
curve = WeightedCurve(space, poly)

# 6. Retrieve the rational points on this curve using the curve.rational_points() method.
rational_points = curve.rational_points()

# 7. Set the divisor_degree to 0.
divisor_degree = 0

# 8. Generate the evaluation code by calling curve.evaluation_code() with the rational_points and divisor_degree.
eval_code = curve.evaluation_code(rational_points, divisor_degree)

# 9. Verify self-orthogonality (should be True for this example)
if eval_code.is_self_orthogonal():
    print(f'Classical code is self-orthogonal: {eval_code.is_self_orthogonal()}')
    # 10. Construct the CSS quantum code
    quantum_code_params = construct_css_quantum_code(eval_code)

    # Extract parameters into local variables for robust printing
    n_Q_val = quantum_code_params['n_Q']
    k_Q_val = quantum_code_params['k_Q']
    d_Q_val = quantum_code_params['d_Q']
    C_X_gen_matrix = quantum_code_params['C_X'].generator_matrix
    C_Z_gen_matrix = quantum_code_params['C_Z'].generator_matrix

    # 11. Print the quantum code parameters
    print('\nConstructed CSS Quantum Code Parameters:')
    print(f'  n_Q (code length): {n_Q_val}')
    print(f'  k_Q (logical qubits): {k_Q_val}')
    print(f'  d_Q (minimum distance): {d_Q_val}')
    print(f'  C_X generator matrix:\n{C_X_gen_matrix}')
    print(f'  C_Z generator matrix:\n{C_Z_gen_matrix}')
else:
    print(f'Classical code is NOT self-orthogonal: {eval_code.is_self_orthogonal()}')

"""

## Final Task

### Subtask:
Provide a summary of the implemented quantum code construction function via CSS, its parameters, and any remaining limitations or next steps, particularly regarding the minimum distance calculation.


## Summary:

### Q&A
The implemented `construct_css_quantum_code` function, its parameters, and limitations are summarized below:

*   **Function Description**: The `construct_css_quantum_code` function takes a `CustomLinearCode` object, which is expected to be self-orthogonal, and derives the parameters for a CSS (Calderbank-Shor-Steane) quantum code. It uses the `dual_code` method to identify the necessary classical codes for the CSS construction.
*   **Parameters Calculated**:
    *   `n_Q` (quantum code length): Determined by the length of the input classical code.
    *   `k_Q` (logical qubits): Calculated as the difference between the dimension of the dual code (`C_X`) and the dimension of the original code (`C_Z`), i.e., `dim(C_X) - dim(C_Z)`.
    *   `d_Q` (minimum distance): This parameter is currently set to `None`.
*   **Limitations and Next Steps**: The main limitation is the absence of a calculation for the quantum minimum distance, `d_Q`. Calculating `d_Q` is complex and requires further implementation of methods to determine the minimum distance of classical codes and then applying the specific rules for CSS codes (e.g., `d_Q = min(d(C_X), d(C_Z^perp))`). A critical next step is to implement robust methods for calculating `d_Q`.

### Data Analysis Key Findings

*   A `dual_code` method was successfully added to the `LinearCode` class (renamed to `CustomLinearCode`) in `/content/qwag/algeom/weighted_curve.py`, enabling the computation of the dual of a linear code using SageMath's functionality.
*   The `construct_css_quantum_code` function was implemented and integrated, taking a self-orthogonal `CustomLinearCode` as input and asserting its self-orthogonality.
*   The function correctly identifies `C_Z` as the input classical code and `C_X` as its dual, leveraging the new `dual_code` method.
*   The quantum code length `n_Q` is set to the classical code length `C_Z.n`.
*   The number of logical qubits `k_Q` is calculated as `C_X.k - C_Z.k`.
*   The quantum minimum distance `d_Q` is explicitly set to `None`, indicating an unimplemented feature.
*   During the process, the original `LinearCode` class was renamed to `CustomLinearCode` to avoid naming conflicts with SageMath's `LinearCode`, which is now imported globally in the module.
*   A test case involving `GF(2)`, weighted projective space with weights `[1, 1]`, and polynomial `x*y` was successfully executed. For this specific scenario, an evaluation code generated with `divisor_degree = 0` was found to be self-orthogonal.
*   The quantum code parameters derived from the test case were: `n_Q = 2`, `k_Q = 0`, and `d_Q = None`.

### Insights or Next Steps

*   **Implement Quantum Minimum Distance Calculation**: The most crucial next step is to develop and integrate methods for calculating the quantum minimum distance `d_Q`, which is a key parameter for evaluating the error-correcting capability of a quantum code. This will likely involve calculating the minimum distances of the underlying classical codes `C_X` and `C_Z`.
*   **Expand Code Construction Capabilities**: Explore and implement the construction of CSS codes from arbitrary classical codes (not necessarily self-orthogonal) by constructing `C_X` and `C_Z` from an input code and its subcodes, enhancing the flexibility of the framework.


# Task
**Reasoning**:
I need to implement the `min_distance` method within the `CustomLinearCode` class and update the `construct_css_quantum_code` function to calculate `d_Q` using this new method. This requires overwriting the `/content/qwag/algeom/weighted_curve.py` file with the modified content.

The `min_distance` method will create a SageMath `LinearCode` object and call its `minimum_distance()` method.
The `construct_css_quantum_code` function will then use `C_Z.min_distance()` and `C_X.dual_code().min_distance()` to calculate `d_Q`.

After updating the file, I will re-run the test cell `07310b01` to verify the functionality.

```python
%%writefile /content/qwag/algeom/weighted_curve.py
from sage.all import gcd, PolynomialRing
from itertools import product
from sage.all import LinearCode # Import SageMath's LinearCode

from .weighted_projective_space import WeightedProjectiveSpace  # Corrected import

# Define our custom LinearCode class
class CustomLinearCode:
    def __init__(self, generator_matrix):
        self.generator_matrix = generator_matrix
        self.n = generator_matrix.ncols()
        self.k = generator_matrix.rank() # Correctly calculate k as the rank of the generator matrix

    def __repr__(self):
        return f"CustomLinearCode with parameters [n={self.n}, k={self.k}]\nGenerator matrix:\n{self.generator_matrix}"

    def is_self_orthogonal(self):
        """
        Checks if the linear code is self-orthogonal, i.e., C subset C^perp.
        This means that every codeword is orthogonal to every other codeword (including itself).
        For a generator matrix G, this is equivalent to G * G^T == 0.
        """
        if self.generator_matrix.is_zero():
            return True

        result_matrix = self.generator_matrix * self.generator_matrix.transpose()
        return result_matrix.is_zero()

    def dual_code(self):
        """
        Returns the dual code (C^perp) of the current linear code.
        The dual code is defined as the set of all vectors that are orthogonal to every codeword in the original code.
        """
        # Create a SageMath LinearCode instance using the globally imported Sage LinearCode
        sage_code = LinearCode(self.generator_matrix)

        # Compute the dual code using SageMath's functionality
        sage_dual_code = sage_code.dual_code()

        # Return a new instance of our custom CustomLinearCode class with the dual code's generator matrix
        return CustomLinearCode(sage_dual_code.generator_matrix())

    def min_distance(self):
        """
        Calculates the minimum distance of the classical linear code using SageMath's built-in function.
        """
        if self.n == 0 or self.k == 0:
            return 0 # Or some appropriate value for an empty/trivial code

        # Create a SageMath LinearCode instance
        sage_code = LinearCode(self.generator_matrix)
        
        # Use SageMath's built-in minimum_distance function
        return sage_code.minimum_distance()


# Helper function for generating weighted monomials (free function in the module)
def _generate_weighted_monomials(poly_ring, weights, max_weighted_degree):
    """
    Generates a list of monomials in `poly_ring` whose weighted degree is less than or equal to
    `max_weighted_degree`.
    """
    monomials = []
    num_vars = len(weights)
    variables = list(poly_ring.gens())

    if max_weighted_degree < 0:
        return []

    def __generate_recursive_exponents(current_exponents_list, current_weighted_degree, var_idx):
        if var_idx == num_vars:
            monomial = poly_ring(1)
            for i in range(num_vars):
                monomial *= variables[i]**current_exponents_list[i]
            monomials.append(monomial)
            return

        if weights[var_idx] == 0:
            max_exponent_for_var = 0
        else:
            max_exponent_for_var = (max_weighted_degree - current_weighted_degree) // weights[var_idx]

        for exponent in range(max_exponent_for_var + 1):
            __generate_recursive_exponents(
                current_exponents_list + [exponent],
                current_weighted_degree + exponent * weights[var_idx],
                var_idx + 1
            )

    __generate_recursive_exponents([], 0, 0)
    return monomials


class WeightedCurve(WeightedProjectiveSpace):
    def __init__(self, space, polynomial):
        """
        Weighted curve defined by a weighted homogeneous polynomial in the space.
        space: WeightedProjectiveSpace instance
        polynomial: weighted homogeneous polynomial (Sage poly)
        """
        super().__init__(space.field, space.weights)
        self.space = space # Store the space object
        self.poly = polynomial

        # Calculate weighted degree and check for weighted homogeneity
        weighted_degrees_of_terms = []
        if not self.poly: # Handle zero polynomial
            raise ValueError("Polynomial cannot be zero.")

        for exponents_tuple, coeff in self.poly.dict().items():
            term_weighted_degree = sum(exponent * weight for exponent, weight in zip(exponents_tuple, self.weights))
            weighted_degrees_of_terms.append(term_weighted_degree)

        if not weighted_degrees_of_terms:
            raise ValueError("Polynomial is empty (should not happen for non-zero poly).")

        deg = weighted_degrees_of_terms[0]
        if not all(d == deg for d in weighted_degrees_of_terms):
            raise ValueError("Polynomial is not weighted homogeneous with given weights.")

        if deg <= 0:
            raise ValueError("Polynomial must have positive weighted degree.")

        self.weighted_degree = deg # Store the weighted degree

        # Coordinate ring is inherited from space
        self.coord_ring = space.coord_ring

    def __repr__(self):
        return f"WeightedCurve in {self.space} defined by {self.poly}"

    def arithmetic_genus(self):
        """
        Computes the arithmetic genus of the curve.
        This formula is applicable for a smooth hypersurface of weighted degree `d`
        in a weighted projective space of dimension `len(self.weights) - 1`.
        Note: This implementation does not account for singularities and provides the arithmetic genus.
        """
        d = self.weighted_degree
        ambient_dim = len(self.weights) - 1 # Dimension of the ambient weighted projective space
        g = 1 + (d - 2) * ambient_dim // 2
        return g

    def smooth_rational_points(self):
        """
        Returns smooth (non-singular) rational points on the curve.
        This implementation iterates over affine points and checks for vanishing
        of the polynomial and its partial derivatives.
        Note: This currently returns affine points. Proper projective normalization
        for weighted projective space is more complex and not implemented here.
        """
        field = self.field
        coord_ring = self.coord_ring
        variables = list(coord_ring.gens())
        num_vars = len(self.weights) # n from (x_0, ..., x_{n-1})

        smooth_points = []

        # Calculate partial derivatives
        partial_derivatives = [self.poly.derivative(var) for var in variables]

        # Iterate over all possible affine points (x_0, ..., x_{num_vars-1})
        for p_coords_tuple in product(field, repeat=num_vars):
            # Create a substitution dictionary
            sub_dict = {var: val for var, val in zip(variables, p_coords_tuple)}

            # 1. Check if the point is on the curve (F(P) = 0)
            if self.poly.subs(sub_dict) == 0:
                # 2. Check for smoothness (not all partial derivatives vanish at P)
                if not self.is_singular(p_coords_tuple):
                    smooth_points.append(p_coords_tuple)

        return smooth_points

    def is_singular(self, p):
        """
        Checks if a given point p (tuple of coordinates) on the curve is singular.
        A point is singular if all partial derivatives of the polynomial vanish at that point.
        """
        coord_ring = self.coord_ring
        variables = list(coord_ring.gens())

        # Create a substitution dictionary for the point p
        sub_dict = {var: val for var, val in zip(variables, p)}

        # Calculate partial derivatives (if not already cached)
        partial_derivatives = [self.poly.derivative(var) for var in variables]

        # Check if all partial derivatives vanish at point p
        for derivative_poly in partial_derivatives:
            if derivative_poly.subs(sub_dict) != 0:
                return False  # Not singular, at least one derivative is non-zero
        return True  # All partial derivatives vanish, so it's singular

    def rational_points(self):
        """
        Returns a list of all distinct rational points on the curve.
        It retrieves all rational points from the ambient weighted projective space
        and filters them to include only those that satisfy the curve's equation.
        """
        curve_points = []
        ambient_points = self.space.rational_points() # Get points from the ambient space

        # Get variables from the coordinate ring to create substitution dictionaries
        variables = list(self.coord_ring.gens())

        for p_coords_tuple in ambient_points:
            # Create a substitution dictionary for the point p
            sub_dict = {var: val for var, val in zip(variables, p_coords_tuple)}

            # Check if the point satisfies the curve's polynomial (F(P) = 0)
            if self.poly.subs(sub_dict) == 0:
                curve_points.append(p_coords_tuple)

        return sorted(curve_points)

    def singular_rational_points(self):
        """
        Returns a list of singular rational points on the curve.
        These are points on the curve where all partial derivatives of the defining
        polynomial vanish simultaneously.
        """
        singular_points = []
        all_rational_points = self.rational_points() # Get all points on the curve

        for p_coords_tuple in all_rational_points:
            if self.is_singular(p_coords_tuple):
                singular_points.append(p_coords_tuple)
        return sorted(singular_points)

    def geometric_genus(self):
        """
        Approximates the geometric genus (p_g) of the curve.
        For a curve with ordinary singularities, the geometric genus can be estimated
        as p_g = p_a - sum(delta_i) where p_a is the arithmetic genus and delta_i
        are contributions from each singularity. For this approximation, we will
        simply subtract the number of singular rational points from the arithmetic genus.

        Disclaimer: This is a highly simplified approximation and does not account
        for the complex nature and multiplicity of singularities. It serves as a
        basic lower bound for the geometric genus for non-singular cases, and
        a rough estimation for singular cases. For precise geometric genus, more
        advanced techniques are required.
        """
        # A very simplified approximation: geometric genus <= arithmetic genus
        # and roughly p_g = p_a - number_of_singular_points
        # This assumes each singular point reduces the genus by 1, which is not generally true
        # for all types of singularities or for non-ordinary singularities.
        num_singular_points = len(self.singular_rational_points())
        approx_geometric_genus = self.arithmetic_genus() - num_singular_points
        return max(0, approx_geometric_genus) # Genus cannot be negative

    def riemann_roch_basis(self, divisor_degree):
        """
        Computes a basis for the Riemann-Roch space L(D) for a divisor D of degree `divisor_degree`.
        """
        if divisor_degree < 0:
            return []
        if divisor_degree == 0:
            return [self.coord_ring(1)] # Space of constant functions

        # Generate candidate monomials up to the desired weighted degree
        candidate_monomials = _generate_weighted_monomials(
            self.coord_ring, self.weights, divisor_degree
        )

        # Compute a Groebner basis for the ideal generated by the curve polynomial
        ideal = self.coord_ring.ideal(self.poly)
        gb = ideal.groebner_basis()

        # Reduce all candidate monomials modulo the Groebner basis to find unique representatives
        basis_elements_reduced = set()
        for monomial in candidate_monomials:
            reduced_form = monomial.reduce(gb)
            basis_elements_reduced.add(reduced_form)

        # Sort the unique basis elements for consistent output
        return sorted(list(basis_elements_reduced), key=str)

    def evaluation_code(self, points_list, divisor_degree):
        """
        Creates a generator matrix for a linear code based on evaluating
        functions from a Riemann-Roch space at a set of rational points.
        """
        from sage.all import matrix # Import matrix from sage.all

        basis = self.riemann_roch_basis(divisor_degree)
        gen_matrix = []

        if not basis or not points_list:
            return CustomLinearCode(matrix(self.field, 0, 0)) # Return empty CustomLinearCode

        for f in basis:
            row = []
            for p in points_list:
                variables = list(self.coord_ring.gens())
                sub_dict = {var: val for var, val in zip(variables, p)}
                row.append(f.subs(sub_dict))
            gen_matrix.append(row)

        if not gen_matrix or not gen_matrix[0]:
            return CustomLinearCode(matrix(self.field, 0, 0))

        G_matrix = matrix(self.field, gen_matrix)

        # Return a CustomLinearCode object
        return CustomLinearCode(G_matrix.echelon_form())

# Function to construct CSS quantum code
def construct_css_quantum_code(classical_code):
    """
    Constructs a CSS (Calderbank-Shor-Steane) quantum code from a self-orthogonal classical linear code.

    Args:
        classical_code (CustomLinearCode): A classical linear code object
                                           that is assumed to be self-orthogonal (C \u2286 C^perp).

    Returns:
        dict: A dictionary containing the quantum code parameters (n_Q, k_Q, d_Q).
    """
    assert classical_code.is_self_orthogonal(), "The classical code must be self-orthogonal for CSS construction."

    C_Z = classical_code
    C_X = classical_code.dual_code()

    n_Q = C_Z.n
    k_Q = C_X.k - C_Z.k

    # Calculate d_Q using the minimum_distance method for both C_Z and C_X.dual_code()
    # d_Q = min(d(C_Z), d(C_X^perp)) -- which is d_Q = min(d(C_Z), d(C)) as C_X = C^perp
    # A common formula for CSS codes is d_Q = min(d(C_Z), d(C_X_perp))
    # where C_X_perp is the code for X-stabilizers.
    # In our case C_X is already the dual of C_Z.
    # The minimum distance of a CSS code [n, k, d]_Q is min(d(C_Z), d(C_X)).
    d_Q = min(C_Z.min_distance(), C_X.min_distance())


    return {
        'n_Q': n_Q,
        'k_Q': k_Q,
        'd_Q': d_Q,
        'C_X': C_X,
        'C_Z': C_Z
    }
```

**Verify the update**:

```python
%%bash
cat /content/qwag/algeom/weighted_curve.py | grep "def min_distance"
cat /content/qwag/algeom/weighted_curve.py | grep -A 5 "def construct_css_quantum_code"
```

## Add Minimum Distance Method to CustomLinearCode

### Subtask:
Add a `min_distance` method to the `CustomLinearCode` class in `/content/qwag/algeom/weighted_curve.py`. This method will use SageMath's built-in `minimum_distance()` function for linear codes to calculate the minimum distance of the classical code. This is a prerequisite for calculating `d_Q`.


**Reasoning**:
I need to add a `min_distance` method to the `CustomLinearCode` class in `/content/qwag/algeom/weighted_curve.py`. This method will use SageMath's built-in `minimum_distance()` function. I will overwrite the entire file using `%%writefile` to include this new method and handle edge cases for zero length or dimension by returning 0.



In [ ]:
%%writefile /content/qwag/algeom/weighted_curve.py
from sage.all import gcd, PolynomialRing
from itertools import product
from sage.all import LinearCode # Import SageMath's LinearCode

from .weighted_projective_space import WeightedProjectiveSpace  # Corrected import

# Define our custom LinearCode class
class CustomLinearCode:
    def __init__(self, generator_matrix):
        self.generator_matrix = generator_matrix
        self.n = generator_matrix.ncols()
        self.k = generator_matrix.rank() # Correctly calculate k as the rank of the generator matrix

    def __repr__(self):
        return f"CustomLinearCode with parameters [n={self.n}, k={self.k}]\nGenerator matrix:\n{self.generator_matrix}"

    def is_self_orthogonal(self):
        """
        Checks if the linear code is self-orthogonal, i.e., C subset C^perp.
        This means that every codeword is orthogonal to every other codeword (including itself).
        For a generator matrix G, this is equivalent to G * G^T == 0.
        """
        if self.generator_matrix.is_zero():
            return True

        result_matrix = self.generator_matrix * self.generator_matrix.transpose()
        return result_matrix.is_zero()

    def dual_code(self):
        """
        Returns the dual code (C^perp) of the current linear code.
        The dual code is defined as the set of all vectors that are orthogonal to every codeword in the original code.
        """
        # Create a SageMath LinearCode instance using the globally imported Sage LinearCode
        sage_code = LinearCode(self.generator_matrix)

        # Compute the dual code using SageMath's functionality
        sage_dual_code = sage_code.dual_code()

        # Return a new instance of our custom CustomLinearCode class with the dual code's generator matrix
        return CustomLinearCode(sage_dual_code.generator_matrix())

    def min_distance(self):
        """
        Calculates the minimum distance of the classical linear code.
        Uses SageMath's built-in `minimum_distance()` function.
        Returns 0 if the code has zero length or dimension.
        """
        if self.n == 0 or self.k == 0:
            return 0
        # Create a SageMath LinearCode instance from our generator matrix
        sage_code = LinearCode(self.generator_matrix)
        return sage_code.minimum_distance()

# Helper function for generating weighted monomials (free function in the module)
def _generate_weighted_monomials(poly_ring, weights, max_weighted_degree):
    """
    Generates a list of monomials in `poly_ring` whose weighted degree is less than or equal to
    `max_weighted_degree`.
    """
    monomials = []
    num_vars = len(weights)
    variables = list(poly_ring.gens())

    if max_weighted_degree < 0:
        return []

    def __generate_recursive_exponents(current_exponents_list, current_weighted_degree, var_idx):
        if var_idx == num_vars:
            monomial = poly_ring(1)
            for i in range(num_vars):
                monomial *= variables[i]**current_exponents_list[i]
            monomials.append(monomial)
            return

        if weights[var_idx] == 0:
            max_exponent_for_var = 0
        else:
            max_exponent_for_var = (max_weighted_degree - current_weighted_degree) // weights[var_idx]

        for exponent in range(max_exponent_for_var + 1):
            __generate_recursive_exponents(
                current_exponents_list + [exponent],
                current_weighted_degree + exponent * weights[var_idx],
                var_idx + 1
            )

    __generate_recursive_exponents([], 0, 0)
    return monomials


class WeightedCurve(WeightedProjectiveSpace):
    def __init__(self, space, polynomial):
        """
        Weighted curve defined by a weighted homogeneous polynomial in the space.
        space: WeightedProjectiveSpace instance
        polynomial: weighted homogeneous polynomial (Sage poly)
        """
        super().__init__(space.field, space.weights)
        self.space = space # Store the space object
        self.poly = polynomial

        # Calculate weighted degree and check for weighted homogeneity
        weighted_degrees_of_terms = []
        if not self.poly: # Handle zero polynomial
            raise ValueError("Polynomial cannot be zero.")

        for exponents_tuple, coeff in self.poly.dict().items():
            term_weighted_degree = sum(exponent * weight for exponent, weight in zip(exponents_tuple, self.weights))
            weighted_degrees_of_terms.append(term_weighted_degree)

        if not weighted_degrees_of_terms:
            raise ValueError("Polynomial is empty (should not happen for non-zero poly).")

        deg = weighted_degrees_of_terms[0]
        if not all(d == deg for d in weighted_degrees_of_terms):
            raise ValueError("Polynomial is not weighted homogeneous with given weights.")

        if deg <= 0:
            raise ValueError("Polynomial must have positive weighted degree.")

        self.weighted_degree = deg # Store the weighted degree

        # Coordinate ring is inherited from space
        self.coord_ring = space.coord_ring

    def __repr__(self):
        return f"WeightedCurve in {self.space} defined by {self.poly}"

    def arithmetic_genus(self):
        """
        Computes the arithmetic genus of the curve.
        This formula is applicable for a smooth hypersurface of weighted degree `d`
        in a weighted projective space of dimension `len(self.weights) - 1`.
        Note: This implementation does not account for singularities and provides the arithmetic genus.
        """
        d = self.weighted_degree
        ambient_dim = len(self.weights) - 1 # Dimension of the ambient weighted projective space
        g = 1 + (d - 2) * ambient_dim // 2
        return g

    def smooth_rational_points(self):
        """
        Returns smooth (non-singular) rational points on the curve.
        This implementation iterates over affine points and checks for vanishing
        of the polynomial and its partial derivatives.
        Note: This currently returns affine points. Proper projective normalization
        for weighted projective space is more complex and not implemented here.
        """
        field = self.field
        coord_ring = self.coord_ring
        variables = list(coord_ring.gens())
        num_vars = len(self.weights) # n from (x_0, ..., x_{n-1})

        smooth_points = []

        # Calculate partial derivatives
        partial_derivatives = [self.poly.derivative(var) for var in variables]

        # Iterate over all possible affine points (x_0, ..., x_{num_vars-1})
        for p_coords_tuple in product(field, repeat=num_vars):
            # Create a substitution dictionary
            sub_dict = {var: val for var, val in zip(variables, p_coords_tuple)}

            # 1. Check if the point is on the curve (F(P) = 0)
            if self.poly.subs(sub_dict) == 0:
                # 2. Check for smoothness (not all partial derivatives vanish at P)
                if not self.is_singular(p_coords_tuple):
                    smooth_points.append(p_coords_tuple)

        return smooth_points

    def is_singular(self, p):
        """
        Checks if a given point p (tuple of coordinates) on the curve is singular.
        A point is singular if all partial derivatives of the polynomial vanish at that point.
        """
        coord_ring = self.coord_ring
        variables = list(coord_ring.gens())

        # Create a substitution dictionary for the point p
        sub_dict = {var: val for var, val in zip(variables, p)}

        # Calculate partial derivatives (if not already cached)
        partial_derivatives = [self.poly.derivative(var) for var in variables]

        # Check if all partial derivatives vanish at point p
        for derivative_poly in partial_derivatives:
            if derivative_poly.subs(sub_dict) != 0:
                return False  # Not singular, at least one derivative is non-zero
        return True  # All partial derivatives vanish, so it's singular

    def rational_points(self):
        """
        Returns a list of all distinct rational points on the curve.
        It retrieves all rational points from the ambient weighted projective space
        and filters them to include only those that satisfy the curve's equation.
        """
        curve_points = []
        ambient_points = self.space.rational_points() # Get points from the ambient space

        # Get variables from the coordinate ring to create substitution dictionaries
        variables = list(self.coord_ring.gens())

        for p_coords_tuple in ambient_points:
            # Create a substitution dictionary for the point p
            sub_dict = {var: val for var, val in zip(variables, p_coords_tuple)}

            # Check if the point satisfies the curve's polynomial (F(P) = 0)
            if self.poly.subs(sub_dict) == 0:
                curve_points.append(p_coords_tuple)

        return sorted(curve_points)

    def singular_rational_points(self):
        """
        Returns a list of singular rational points on the curve.
        These are points on the curve where all partial derivatives of the defining
        polynomial vanish simultaneously.
        """
        singular_points = []
        all_rational_points = self.rational_points() # Get all points on the curve

        for p_coords_tuple in all_rational_points:
            if self.is_singular(p_coords_tuple):
                singular_points.append(p_coords_tuple)
        return sorted(singular_points)

    def geometric_genus(self):
        """
        Approximates the geometric genus (p_g) of the curve.
        For a curve with ordinary singularities, the geometric genus can be estimated
        as p_g = p_a - sum(delta_i) where p_a is the arithmetic genus and delta_i
        are contributions from each singularity. For this approximation, we will
        simply subtract the number of singular rational points from the arithmetic genus.

        Disclaimer: This is a highly simplified approximation and does not account
        for the complex nature and multiplicity of singularities. It serves as a
        basic lower bound for the geometric genus for non-singular cases, and
        a rough estimation for singular cases. For precise geometric genus, more
        advanced techniques are required.
        """
        # A very simplified approximation: geometric genus <= arithmetic genus
        # and roughly p_g = p_a - number_of_singular_points
        # This assumes each singular point reduces the genus by 1, which is not generally true
        # for all types of singularities or for non-ordinary singularities.
        num_singular_points = len(self.singular_rational_points())
        approx_geometric_genus = self.arithmetic_genus() - num_singular_points
        return max(0, approx_geometric_genus) # Genus cannot be negative

    def riemann_roch_basis(self, divisor_degree):
        """
        Computes a basis for the Riemann-Roch space L(D) for a divisor D of degree `divisor_degree`.
        """
        if divisor_degree < 0:
            return []
        if divisor_degree == 0:
            return [self.coord_ring(1)] # Space of constant functions

        # Generate candidate monomials up to the desired weighted degree
        candidate_monomials = _generate_weighted_monomials(
            self.coord_ring, self.weights, divisor_degree
        )

        # Compute a Groebner basis for the ideal generated by the curve polynomial
        ideal = self.coord_ring.ideal(self.poly)
        gb = ideal.groebner_basis()

        # Reduce all candidate monomials modulo the Groebner basis to find unique representatives
        basis_elements_reduced = set()
        for monomial in candidate_monomials:
            reduced_form = monomial.reduce(gb)
            basis_elements_reduced.add(reduced_form)

        # Sort the unique basis elements for consistent output
        return sorted(list(basis_elements_reduced), key=str)

    def evaluation_code(self, points_list, divisor_degree):
        """
        Creates a generator matrix for a linear code based on evaluating
        functions from a Riemann-Roch space at a set of rational points.
        """
        from sage.all import matrix # Import matrix from sage.all

        basis = self.riemann_roch_basis(divisor_degree)
        gen_matrix = []

        if not basis or not points_list:
            return CustomLinearCode(matrix(self.field, 0, 0)) # Return empty CustomLinearCode

        for f in basis:
            row = []
            for p in points_list:
                variables = list(self.coord_ring.gens())
                sub_dict = {var: val for var, val in zip(variables, p)}
                row.append(f.subs(sub_dict))
            gen_matrix.append(row)

        if not gen_matrix or not gen_matrix[0]:
            return CustomLinearCode(matrix(self.field, 0, 0))

        G_matrix = matrix(self.field, gen_matrix)

        # Return a CustomLinearCode object
        return CustomLinearCode(G_matrix.echelon_form())

# Function to construct CSS quantum code
def construct_css_quantum_code(classical_code):
    """
    Constructs a CSS (Calderbank-Shor-Steane) quantum code from a self-orthogonal classical linear code.
    """
    assert classical_code.is_self_orthogonal(), "The classical code must be self-orthogonal for CSS construction."

    C_Z = classical_code
    C_X = classical_code.dual_code()

    n_Q = C_Z.n
    k_Q = C_X.k - C_Z.k
    d_Q = None

    return {
        'n_Q': n_Q,
        'k_Q': k_Q,
        'd_Q': d_Q,
        'C_X': C_X,
        'C_Z': C_Z
    }

**Reasoning**:
The previous step successfully added the `min_distance` method to the `CustomLinearCode` class. Now, as instructed, I will verify its addition by grepping for the method definition in the `weighted_curve.py` file.



In [ ]:
%%bash
cat /content/qwag/algeom/weighted_curve.py | grep "def min_distance"


**Reasoning**:
Now that the `min_distance` method has been successfully added to `CustomLinearCode`, I need to update the `construct_css_quantum_code` function to utilize this new method for calculating `d_Q`. This requires overwriting the entire `weighted_curve.py` file with the updated `construct_css_quantum_code` function.



In [ ]:
%%writefile /content/qwag/algeom/weighted_curve.py
from sage.all import gcd, PolynomialRing
from itertools import product
from sage.all import LinearCode # Import SageMath's LinearCode

from .weighted_projective_space import WeightedProjectiveSpace  # Corrected import

# Define our custom LinearCode class
class CustomLinearCode:
    def __init__(self, generator_matrix):
        self.generator_matrix = generator_matrix
        self.n = generator_matrix.ncols()
        self.k = generator_matrix.rank() # Correctly calculate k as the rank of the generator matrix

    def __repr__(self):
        return f"""CustomLinearCode with parameters [n={self.n}, k={self.k}]
Generator matrix:
{self.generator_matrix}"""

    def is_self_orthogonal(self):
        """
        Checks if the linear code is self-orthogonal, i.e., C subset C^perp.
        This means that every codeword is orthogonal to every other codeword (including itself).
        For a generator matrix G, this is equivalent to G * G^T == 0.
        """
        if self.generator_matrix.is_zero():
            return True

        result_matrix = self.generator_matrix * self.generator_matrix.transpose()
        return result_matrix.is_zero()

    def dual_code(self):
        """
        Returns the dual code (C^perp) of the current linear code.
        The dual code is defined as the set of all vectors that are orthogonal to every codeword in the original code.
        """
        if self.n == 0:
            # If the code has length 0, its dual also has length 0 and dimension 0
            from sage.all import matrix
            return CustomLinearCode(matrix(self.generator_matrix.base_ring(), 0, 0))

        # Create a SageMath LinearCode instance using the globally imported Sage LinearCode
        sage_code = LinearCode(self.generator_matrix)

        # Compute the dual code using SageMath's functionality
        sage_dual_code = sage_code.dual_code()

        # Return a new instance of our custom CustomLinearCode class with the dual code's generator matrix
        return CustomLinearCode(sage_dual_code.generator_matrix())

    def min_distance(self):
        """
        Calculates the minimum distance of the classical linear code.
        Uses SageMath's built-in `minimum_distance()` function.
        Returns 0 if the code has zero length or dimension.
        """
        if self.n == 0 or self.k == 0:
            return 0
        # Create a SageMath LinearCode instance from our generator matrix
        sage_code = LinearCode(self.generator_matrix)
        return sage_code.minimum_distance()

# Helper function for calculating the total weighted degree of a polynomial
def total_weighted_degree(polynomial, weights):
    """
    Computes the total weighted degree of a weighted homogeneous polynomial.
    Returns the common weighted degree of all terms if the polynomial is homogeneous.
    Raises ValueError if the polynomial is not weighted homogeneous.
    """
    weighted_degrees_of_terms = []
    if not polynomial: # Handle zero polynomial
        # Conventionally, the degree of the zero polynomial is -infinity or undefined.
        # For our purposes here, we might return 0 or raise an error depending on context.
        # Let's return 0 as it usually means a trivial case in degrees.
        return 0

    for exponents_tuple, coeff in polynomial.dict().items():
        term_weighted_degree = sum(exponent * weight for exponent, weight in zip(exponents_tuple, weights))
        weighted_degrees_of_terms.append(term_weighted_degree)

    if not weighted_degrees_of_terms:
        return 0 # Should not happen for non-zero polynomial based on above check

    deg = weighted_degrees_of_terms[0]
    if not all(d == deg for d in weighted_degrees_of_terms):
        raise ValueError("Polynomial is not weighted homogeneous with given weights.")

    return deg

# Helper function for generating weighted monomials (free function in the module)
def _generate_weighted_monomials(poly_ring, weights, max_weighted_degree):
    """
    Generates a list of monomials in `poly_ring` whose weighted degree is less than or equal to
    `max_weighted_degree`.
    """
    monomials = []
    num_vars = len(weights)
    variables = list(poly_ring.gens())

    if max_weighted_degree < 0:
        return []

    def __generate_recursive_exponents(current_exponents_list, current_weighted_degree, var_idx):
        if var_idx == num_vars:
            monomial = poly_ring(1)
            for i in range(num_vars):
                monomial *= variables[i]**current_exponents_list[i]
            monomials.append(monomial)
            return

        if weights[var_idx] == 0:
            max_exponent_for_var = 0
        else:
            max_exponent_for_var = (max_weighted_degree - current_weighted_degree) // weights[var_idx]

        for exponent in range(max_exponent_for_var + 1):
            __generate_recursive_exponents(
                current_exponents_list + [exponent],
                current_weighted_degree + exponent * weights[var_idx],
                var_idx + 1
            )

    __generate_recursive_exponents([], 0, 0)
    return monomials


class WeightedCurve(WeightedProjectiveSpace):
    def __init__(self, space, polynomial):
        """
        Weighted curve defined by a weighted homogeneous polynomial in the space.
        space: WeightedProjectiveSpace instance
        polynomial: weighted homogeneous polynomial (Sage poly)
        """
        super().__init__(space.field, space.weights)
        self.space = space # Store the space object
        self.poly = polynomial

        # Calculate weighted degree and check for weighted homogeneity
        try:
            deg = total_weighted_degree(self.poly, self.weights)
        except ValueError as e:
            raise ValueError(f"Polynomial is not weighted homogeneous: {e}") from e

        if deg <= 0:
            raise ValueError("Polynomial must have positive weighted degree.")

        self.weighted_degree = deg # Store the weighted degree

        # Coordinate ring is inherited from space
        self.coord_ring = space.coord_ring

    def __repr__(self):
        return f"WeightedCurve in {self.space} defined by {self.poly}"

    def arithmetic_genus(self):
        """
        Computes the arithmetic genus of the curve.
        This formula is applicable for a smooth hypersurface of weighted degree `d`
        in a weighted projective space of dimension `len(self.weights) - 1`.
        Note: This implementation does not account for singularities and provides the arithmetic genus.
        """
        d = self.weighted_degree
        ambient_dim = len(self.weights) - 1 # Dimension of the ambient weighted projective space
        g = 1 + (d - 2) * ambient_dim // 2
        return g

    def smooth_rational_points(self):
        """
        Returns smooth (non-singular) rational points on the curve.
        This implementation filters the rational points on the curve
        to include only those that are non-singular in the projective sense.
        """
        smooth_points = []
        all_curve_rational_points = self.rational_points() # Get all projective rational points on the curve

        for p_coords_tuple in all_curve_rational_points:
            # Check for smoothness (not all partial derivatives vanish at P)
            if not self.is_singular(p_coords_tuple):
                smooth_points.append(p_coords_tuple)

        return smooth_points

    def is_singular(self, p):
        """
        Checks if a given point p (tuple of coordinates) on the curve is singular.
        A point is singular if all partial derivatives of the polynomial vanish at that point.
        """
        coord_ring = self.coord_ring
        variables = list(coord_ring.gens())

        # Create a substitution dictionary for the point p
        sub_dict = {var: val for var, val in zip(variables, p)}

        # Calculate partial derivatives (if not already cached)
        partial_derivatives = [self.poly.derivative(var) for var in variables]

        # Check if all partial derivatives vanish at point p
        for derivative_poly in partial_derivatives:
            if derivative_poly.subs(sub_dict) != 0:
                return False  # Not singular, at least one derivative is non-zero
        return True  # All partial derivatives vanish, so it's singular

    def rational_points(self):
        """
        Returns a list of all distinct rational points on the curve.
        It retrieves all rational points from the ambient weighted projective space
        and filters them to include only those that satisfy the curve's equation.
        """
        curve_points = []
        ambient_points = self.space.rational_points() # Get points from the ambient space

        # Get variables from the coordinate ring to create substitution dictionaries
        variables = list(self.coord_ring.gens())

        for p_coords_tuple in ambient_points:
            # Create a substitution dictionary for the point p
            sub_dict = {var: val for var, val in zip(variables, p_coords_tuple)}

            # Check if the point satisfies the curve's polynomial (F(P) = 0)
            if self.poly.subs(sub_dict) == 0:
                curve_points.append(p_coords_tuple)

        return sorted(curve_points)

    def singular_rational_points(self):
        """
        Returns a list of singular rational points on the curve.
        These are points on the curve where all partial derivatives of the defining
        polynomial vanish simultaneously.
        """
        singular_points = []
        all_rational_points = self.rational_points() # Get all points on the curve

        for p_coords_tuple in all_rational_points:
            if self.is_singular(p_coords_tuple):
                singular_points.append(p_coords_tuple)
        return sorted(singular_points)

    def geometric_genus(self):
        """
        Approximates the geometric genus (p_g) of the curve.
        For a curve with ordinary singularities, the geometric genus can be estimated
        as p_g = p_a - sum(delta_i) where p_a is the arithmetic genus and delta_i
        are contributions from each singularity. For this approximation, we will
        simply subtract the number of singular rational points from the arithmetic genus.

        Disclaimer: This is a highly simplified approximation and does not account
        for the complex nature and multiplicity of singularities. It serves as a
        basic lower bound for the geometric genus for non-singular cases, and
        a rough estimation for singular cases. For precise geometric genus, more
        advanced techniques are required.
        """
        # A very simplified approximation: geometric genus <= arithmetic genus
        # and roughly p_g = p_a - number_of_singular_points
        # This assumes each singular point reduces the genus by 1, which is not generally true
        # for all types of singularities or for non-ordinary singularities.
        num_singular_points = len(self.singular_rational_points())
        approx_geometric_genus = self.arithmetic_genus() - num_singular_points
        return max(0, approx_geometric_genus) # Genus cannot be negative

    def riemann_roch_basis(self, divisor_degree):
        """
        Computes a basis for the Riemann-Roch space L(D) for a divisor D of degree `divisor_degree`.
        """
        if divisor_degree < 0:
            return []
        if divisor_degree == 0:
            return [self.coord_ring(1)] # Space of constant functions

        # Generate candidate monomials up to the desired weighted degree
        candidate_monomials = _generate_weighted_monomials(
            self.coord_ring, self.weights, divisor_degree
        )

        # Compute a Groebner basis for the ideal generated by the curve polynomial
        ideal = self.coord_ring.ideal(self.poly)
        gb = ideal.groebner_basis()

        # Reduce all candidate monomials modulo the Groebner basis to find unique representatives
        basis_elements_reduced = set()
        for monomial in candidate_monomials:
            reduced_form = monomial.reduce(gb)
            basis_elements_reduced.add(reduced_form)

        # Sort the unique basis elements for consistent output
        return sorted(list(basis_elements_reduced), key=str)

    def evaluation_code(self, points_list, divisor_degree):
        """
        Creates a generator matrix for a linear code based on evaluating
        functions from a Riemann-Roch space at a set of rational points.
        """
        from sage.all import matrix # Import matrix from sage.all

        basis = self.riemann_roch_basis(divisor_degree)
        gen_matrix = []

        if not basis or not points_list:
            return CustomLinearCode(matrix(self.field, 0, 0)) # Return empty CustomLinearCode

        for f in basis:
            row = []
            for p in points_list:
                variables = list(self.coord_ring.gens())
                sub_dict = {var: val for var, val in zip(variables, p)}
                row.append(f.subs(sub_dict))
            gen_matrix.append(row)

        if not gen_matrix or not gen_matrix[0]:
            return CustomLinearCode(matrix(self.field, 0, 0))

        G_matrix = matrix(self.field, gen_matrix)

        # Return a CustomLinearCode object
        return CustomLinearCode(G_matrix.echelon_form())

def refined_singleton_bound(n, k, epsilon=0):
    """
    Refined quantum Singleton bound from the paper:
    d ≤ (n - k + 2)/2 - ε/2, where ε > 0 from singularities.
    """
    return (n - k + 2) // 2 - epsilon // 2

# Function to construct CSS quantum code
def construct_css_quantum_code(classical_code, curve=None):
    """
    Constructs a CSS (Calderbank-Shor-Steane) quantum code from a self-orthogonal classical linear code.

    Args:
        classical_code (CustomLinearCode): A classical linear code object
                                           that is assumed to be self-orthogonal (C ⊆ C^perp).
        curve (WeightedCurve, optional): The WeightedCurve object from which the classical code was derived.
                                         Used to estimate epsilon for the refined Singleton bound.

    Returns:
        dict: A dictionary containing the quantum code parameters (n_Q, k_Q, d_Q, d_Q_refined_bound).
    """
    assert classical_code.is_self_orthogonal(), "The classical code must be self-orthogonal for CSS construction."

    C_Z = classical_code
    C_X = classical_code.dual_code()

    n_Q = C_Z.n
    k_Q = C_X.k - C_Z.k

    # Calculate d_Q using the minimum_distance method for both C_Z and C_X
    # The minimum distance of a CSS code [n, k, d]_Q is min(d(C_Z), d(C_X)).
    d_Q = min(C_Z.min_distance(), C_X.min_distance())

    # Calculate refined quantum Singleton bound
    epsilon_estimate = 0
    if curve is not None:
        # Simple heuristic: epsilon is the number of singular rational points
        epsilon_estimate = len(curve.singular_rational_points())

    d_Q_refined_bound = refined_singleton_bound(n_Q, k_Q, epsilon_estimate)


    return {
        'n_Q': n_Q,
        'k_Q': k_Q,
        'd_Q': d_Q,
        'd_Q_refined_bound': d_Q_refined_bound,
        'C_X': C_X,
        'C_Z': C_Z
    }

**Reasoning**:
Now that the `min_distance` method has been successfully added to `CustomLinearCode` and the `construct_css_quantum_code` function has been updated to use it for calculating `d_Q`, I will re-run the test cell `07310b01` to verify the functionality.



In [ ]:
%%bash
source activate sage
export PYTHONPATH=$PYTHONPATH:/content
sage -c """
from sage.all import GF
from qwag.algeom.weighted_projective_space import WeightedProjectiveSpace
from qwag.algeom.weighted_curve import WeightedCurve, CustomLinearCode, construct_css_quantum_code

# 1. Define the finite field as GF(2).
field = GF(2) # Changed to GF(2) for a known self-orthogonal example

# 2. Define a weighted projective space using the GF(2) field and weights [1, 1].
weights = [1, 1] # Changed to [1,1] for a known self-orthogonal example
space = WeightedProjectiveSpace(field, weights)

# 3. Get the variables (x, y) from the coordinate ring of the defined space.
x, y = space.coord_ring.gens()

# 4. Define a weighted homogeneous polynomial poly = x*y.
poly = x*y # Changed to x*y for a known self-orthogonal example

# 5. Instantiate the WeightedCurve class with the created space and the polynomial.
curve = WeightedCurve(space, poly)

# 6. Retrieve the rational points on this curve using the curve.rational_points() method.
rational_points = curve.rational_points()

# 7. Set the divisor_degree to 0.
divisor_degree = 0

# 8. Generate the evaluation code by calling curve.evaluation_code() with the rational_points and divisor_degree.
eval_code = curve.evaluation_code(rational_points, divisor_degree)

# 9. Verify self-orthogonality (should be True for this example)
if eval_code.is_self_orthogonal():
    print(f'Classical code is self-orthogonal: {eval_code.is_self_orthogonal()}')
    # 10. Construct the CSS quantum code, passing the curve object
    quantum_code_params = construct_css_quantum_code(eval_code, curve=curve)

    # Extract parameters into local variables for robust printing
    n_Q_val = quantum_code_params['n_Q']
    k_Q_val = quantum_code_params['k_Q']
    d_Q_val = quantum_code_params['d_Q']
    d_Q_refined_bound_val = quantum_code_params['d_Q_refined_bound'] # New parameter
    C_X_gen_matrix = quantum_code_params['C_X'].generator_matrix
    C_Z_gen_matrix = quantum_code_params['C_Z'].generator_matrix

    # 11. Print the quantum code parameters
    print('\nConstructed CSS Quantum Code Parameters:')
    print(f'  n_Q (code length): {n_Q_val}')
    print(f'  k_Q (logical qubits): {k_Q_val}')
    print(f'  d_Q (minimum distance): {d_Q_val}')
    print(f'  d_Q (refined Singleton bound): {d_Q_refined_bound_val}') # Print new bound
    print(f'  C_X generator matrix:\n{C_X_gen_matrix}')
    print(f'  C_Z generator matrix:\n{C_Z_gen_matrix}')
else:
    print(f'Classical code is NOT self-orthogonal: {eval_code.is_self_orthogonal()}')

"""

In [ ]:
%%bash
source activate sage
export PYTHONPATH=$PYTHONPATH:/content
sage -c """
from qwag.algeom.weighted_projective_space import WeightedProjectiveSpace
from qwag.algeom.weighted_curve import WeightedCurve, CustomLinearCode, construct_css_quantum_code
from sage.all import GF

# Define the finite field
field = GF(7) # You can change this finite field

# Define the weights: wt(x)=1, wt(y)=3, wt(z)=1
weights = [1, 3, 1]
space = WeightedProjectiveSpace(field, weights)

# Get the variables from the coordinate ring
x, y, z = space.coord_ring.gens()

# Define the weighted homogeneous polynomial
# y^2 = x^6 + a*x^4*z^2 + b*x^2*z^4 + c*z^6
# Here, we choose specific values for a, b, c. You can change them.
a_param = field(1)
b_param = field(1)
c_param = field(1)
poly = y**2 - (x**6 + a_param*x**4*z**2 + b_param*x**2*z**4 + c_param*z**6)

curve = WeightedCurve(space, poly)

print(f'Curve: {curve}')
print(f'Arithmetic Genus: {curve.arithmetic_genus()}')
print(f'Singular Rational Points: {curve.singular_rational_points()}')
print(f'Geometric Genus (approx): {curve.geometric_genus()}')

# Attempt to generate an evaluation code and check self-orthogonality
rational_points = curve.rational_points()
print(f'Rational points on curve: {rational_points}')

# Choose a divisor degree (e.g., the weighted degree of the polynomial)
divisor_degree = curve.weighted_degree # Which is 6 in this case

eval_code = curve.evaluation_code(rational_points, divisor_degree)

if eval_code.n > 0: # Only check if the code is not empty
    print(f'\nEvaluation Code (Divisor Degree {divisor_degree}):')
    print(eval_code)
    is_self_orthogonal = eval_code.is_self_orthogonal()
    print(f'Is the Linear Code self-orthogonal? {is_self_orthogonal}')

    if is_self_orthogonal:
        quantum_code_params = construct_css_quantum_code(eval_code, curve=curve)
        print('\nConstructed CSS Quantum Code Parameters:')
        print(f'  n_Q (code length): {quantum_code_params["n_Q"]}')
        print(f'  k_Q (logical qubits): {quantum_code_params["k_Q"]}')
        print(f'  d_Q (minimum distance): {quantum_code_params["d_Q"]}')
        print(f'  d_Q (refined Singleton bound): {quantum_code_params["d_Q_refined_bound"]}')
        print(f'  C_X generator matrix:\n{quantum_code_params["C_X"].generator_matrix}')
        print(f'  C_Z generator matrix:\n{quantum_code_params["C_Z"].generator_matrix}')
    else:
        print('Cannot construct CSS quantum code: Classical code is not self-orthogonal.')
else:
    print('\nNo rational points on the curve, so no evaluation code can be formed.')
"""

In [ ]:
%%bash
source activate sage
export PYTHONPATH=$PYTHONPATH:/content
sage -c """
from sage.all import GF, matrix, gcd, PolynomialRing
from qwag.algeom.weighted_projective_space import WeightedProjectiveSpace
from qwag.algeom.weighted_curve import WeightedCurve, CustomLinearCode, construct_css_quantum_code

# 1. Define the weighted space
field = GF(7)
weights = [2, 5, 2]  # wt(x)=2, wt(y)=5, wt(z)=2
space = WeightedProjectiveSpace(field, weights)

# 2. Coordinate ring and variables (inherited)
x, y, z = space.coord_ring.gens()  # x0=x, x1=y, x2=z

# 3. Define the weighted homogeneous polynomial (degree 10)
poly = y**2 - x**5 - 1 * x**3 * z**2 # Corrected to be weighted homogeneous of degree 10

# 4. Create the curve
curve = WeightedCurve(space, poly)

# Quick checks
print(curve)
print('Arithmetic genus:', curve.arithmetic_genus())
print('Is well-formed space?', space.is_well_formed())  # gcd=1? No, gcd(2,5,2)=1 → yes

# 5. Get smooth rational points (your method)
smooth_pts = curve.smooth_rational_points()
n = len(smooth_pts)
print(f'Number of smooth rational points: {n}')

# 6. Choose divisor G = 3 * infinity point (or similar)
# The concept of "infinity point" needs careful definition for weighted projective spaces.
# For now, let's just use the divisor degree directly for basis generation as in evaluation_code.
# Using simple divisor degree as used in evaluation_code
divisor_degree_G = 3 # This is deg G

# 7. Evaluation divisor D = sum smooth points
# D.support() is essentially the list of evaluation points.

# 8. Build the classical AG code C = C_L(D, G)
C = curve.evaluation_code(smooth_pts, divisor_degree_G)  # your method

# Handle empty codes gracefully for printing
if C.n > 0:
    print(f'Classical code: [n={C.n}, k={C.k}, d>={C.min_distance()}]')

    # 9. Check self-orthogonality
    if C.is_self_orthogonal():
        print('C is self-orthogonal → CSS ready')

        # 10. Construct quantum code Q via CSS
        quantum_params = construct_css_quantum_code(C, curve=curve)
        print(f'Constructed CSS Quantum Code Parameters:')
        print(f'  n_Q (code length): {quantum_params["n_Q"]}')
        print(f'  k_Q (logical qubits): {quantum_params["k_Q"]}')
        print(f'  d_Q (minimum distance): {quantum_params["d_Q"]}')
        print(f'  d_Q (refined Singleton bound): {quantum_params["d_Q_refined_bound"]}')
        print(f'  C_X generator matrix:\n{quantum_params["C_X"].generator_matrix}')
        print(f'  C_Z generator matrix:\n{quantum_params["C_Z"].generator_matrix}')

    else:
        print('Not self-orthogonal — try different G or curve')
else:
    print('No classical code could be formed (e.g., no smooth points found).')
"""

In [ ]:
%%bash
source activate sage
export PYTHONPATH=$PYTHONPATH:/content

cat << 'EOF_PYTHON_SCRIPT' > /tmp/temp_sage_script.py
from qwag.algeom.weighted_projective_space import WeightedProjectiveSpace
from qwag.algeom.weighted_curve import WeightedCurve, total_weighted_degree
from sage.all import GF, PolynomialRing

# Define the finite field and weights
field = GF(7)
weights = [1, 3, 1]  # wt(x)=1, wt(y)=3, wt(z)=1

# Define a coordinate ring for x,y,z
coord_ring_xyz = PolynomialRing(field, 'x,y,z')
x, y, z = coord_ring_xyz.gens()

print('--- Testing total_weighted_degree for non-homogeneous polynomial ---')
# This polynomial is NOT weighted homogeneous with respect to weights [1,3,1]
# Term y**2 has degree 3*2 = 6
# Term x**5 has degree 1*5 = 5
# This should trigger a ValueError from total_weighted_degree
poly_non_homogeneous = y**2 + x**5
try:
    deg_non_homo = total_weighted_degree(poly_non_homogeneous, weights)
    print("Weighted degree of '{}' with weights {}: {}".format(poly_non_homogeneous, weights, deg_non_homo))
except ValueError as e:
    print("Caught expected error: {}".format(e))

print('\n--- Testing total_weighted_degree for a homogeneous polynomial and curve genus ---')
# This polynomial IS weighted homogeneous with respect to weights [1,3,1]
# Term y**2 has degree 3*2 = 6
# Term x**6 has degree 1*6 = 6
poly_homogeneous = y**2 - x**6
deg_homo = total_weighted_degree(poly_homogeneous, weights)
print("Weighted degree of '{}' with weights {}: {}".format(poly_homogeneous, weights, deg_homo))

# Create a WeightedProjectiveSpace instance
space = WeightedProjectiveSpace(field, weights)

# Create a WeightedCurve instance using the homogeneous polynomial
curve = WeightedCurve(space, poly_homogeneous)

print('\nWeighted Curve: {}'.format(curve))
print('Arithmetic Genus of the curve: {}'.format(curve.arithmetic_genus()))
EOF_PYTHON_SCRIPT

sage /tmp/temp_sage_script.py


In [ ]:
%%writefile -a /content/qwag/algeom/weighted_curve.py

def is_weighted_superelliptic_curve(space, polynomial):
    """
    Checks if the given weighted projective curve defined by 'polynomial' in 'space'
    is a weighted superelliptic curve, i.e., of the form y^n = f(x0, ..., x_{k-1}, x_{k+1}, ..., xn)
    for some variable y = x_k, n >= 2, where f is weighted homogeneous.

    Parameters:
    - space: WeightedProjectiveSpace instance with field, weights, and coord_ring.
    - polynomial: Sage polynomial defining the curve (must be weighted homogeneous).

    Returns:
    - (bool, dict): True if it is superelliptic, with details {'level': n, 'y_var': y, 'f': f}; False otherwise.
    """
    from sage.all import PolynomialRing, gcd, ZZ, prod # Added prod for prod(v**e for v, e in zip(vars, exp))

    # Get variables and weights
    vars = space.coord_ring.gens()
    weights = space.weights
    n_vars = len(vars)

    # Ensure polynomial is weighted homogeneous
    # This check is more robust if done via total_weighted_degree, which handles poly.monomials() correctly
    try:
        poly_deg = total_weighted_degree(polynomial, weights)
    except ValueError:
        return False, "Polynomial is not weighted homogeneous."

    # Check for each possible y_var = vars[k]
    for k in range(n_vars):
        y = vars[k]
        wt_y = weights[k]

        # Extract terms with y
        # Create a dict that maps variable tuples to coefficients
        terms_dict = polynomial.dict()

        # Find terms that contain y
        y_containing_terms = {exp_tuple: coeff for exp_tuple, coeff in terms_dict.items() if exp_tuple[k] > 0}
        other_terms = {exp_tuple: coeff for exp_tuple, coeff in terms_dict.items() if exp_tuple[k] == 0}

        highest_y_exp = 0
        if y_containing_terms:
            highest_y_exp = max(exp_tuple[k] for exp_tuple in y_containing_terms.keys())

        if highest_y_exp < 2:
            continue  # n must be >= 2 for superelliptic curves

        n_level = highest_y_exp
        expected_degree_f = n_level * wt_y

        # Check if the polynomial can be written as C * y^n_level + f(other vars)
        # We need exactly one term with y^n_level and no other y-containing terms
        if len(y_containing_terms) != 1:
            continue # If there are more than one term containing y, or no terms containing y

        (y_exp_tuple, y_coeff) = list(y_containing_terms.items())[0]

        if y_exp_tuple[k] != n_level:
            continue # This should not happen due to highest_y_exp logic, but for safety

        # The coefficient of y^n_level should be invertible (unit). For simplicity, check 1 or -1 for prime fields.
        # More generally, it should be a unit in the field.
        if y_coeff == 0 or not y_coeff.is_unit(): # or check y_coeff not in [field(1), field(-1)] for simpler check
            continue

        # Construct f = - (polynomial - y_coeff * y^n_level) / y_coeff
        # In other words, f is all other terms, possibly negated and scaled
        f_poly = space.coord_ring.zero()
        for exp_tuple, coeff in other_terms.items():
            monomial_term = prod(var**e for var, e in zip(vars, exp_tuple))
            f_poly += coeff * monomial_term

        # Ensure that f_poly is weighted homogeneous of the expected degree
        try:
            f_deg = total_weighted_degree(f_poly, weights)
            if f_deg != expected_degree_f:
                continue
        except ValueError:
            continue # f_poly is not weighted homogeneous

        # If all checks pass, it's a weighted superelliptic curve of the form y^n = f
        # We need to ensure the original poly matches: y_coeff * y^n_level + f_poly == polynomial
        # which means y_coeff * y^n_level = polynomial - f_poly
        # So, the polynomial should be equivalent to y_coeff * y^n_level + f_poly
        if polynomial == (y_coeff * y**n_level + f_poly):
            # The canonical form is y^n = (-f_poly / y_coeff)
            return True, {'level': n_level, 'y_var': y, 'f': -f_poly / y_coeff}

    return False, "Does not match weighted superelliptic form."


In [ ]:
%%bash
source activate sage
export PYTHONPATH=$PYTHONPATH:/content

cat << 'EOF_PYTHON_SCRIPT' > /tmp/temp_sage_script.py
from qwag.algeom.weighted_projective_space import WeightedProjectiveSpace
from qwag.algeom.weighted_curve import WeightedCurve, is_weighted_superelliptic_curve, total_weighted_degree
from sage.all import GF, PolynomialRing

# Define field and weights
field = GF(7)
weights_case1 = [1, 2, 3] # w_x=1, w_y=2, w_z=3
space_case1 = WeightedProjectiveSpace(field, weights_case1)
x, y, z = space_case1.coord_ring.gens()

print('--- Test Case 1: Weighted Superelliptic Curve (y^2 = x^6 + z^2) ---\n')
# y^2 = x^6 + z^2
# Weighted degree: y^2 -> 2*2 = 4 (wt y is 2, assuming 3 for y makes y^2 degree 6), x^6 -> 1*6=6, z^2 -> 3*2=6
# Let's adjust weights to [1,3,1] for this common example for simplicity.
weights_superelliptic = [1, 3, 1] # w_x=1, w_y=3, w_z=1
space_superelliptic = WeightedProjectiveSpace(field, weights_superelliptic)
x_s, y_s, z_s = space_superelliptic.coord_ring.gens()

poly_superelliptic = y_s**2 - (x_s**6 + z_s**6) # Degree 6 for all terms
curve_superelliptic = WeightedCurve(space_superelliptic, poly_superelliptic)

is_superelliptic, details = is_weighted_superelliptic_curve(space_superelliptic, poly_superelliptic)
print("Curve: {}".format(curve_superelliptic.poly))
print("Weights: {}".format(weights_superelliptic))
print("Is superelliptic? {}".format(is_superelliptic))
if is_superelliptic:
    print("  Details: level={}, y_var={}, f={}".format(details['level'], details['y_var'], details['f']))


print('\n--- Test Case 2: Non-Superelliptic Curve (y^2 + x^3 + z) ---\n')
weights_non_superelliptic = [1, 2, 3] # w_x=1, w_y=2, w_z=3
space_non_superelliptic = WeightedProjectiveSpace(field, weights_non_superelliptic)
x_n, y_n, z_n = space_non_superelliptic.coord_ring.gens()

# This polynomial is weighted homogeneous of degree 6, but not superelliptic form
# x^6 (deg 6), y^3 (deg 6), z^2 (deg 6)
poly_non_superelliptic = x_n**6 + y_n**3 + z_n**2
curve_non_superelliptic = WeightedCurve(space_non_superelliptic, poly_non_superelliptic)

is_superelliptic, details = is_weighted_superelliptic_curve(space_non_superelliptic, poly_non_superelliptic)
print("Curve: {}".format(curve_non_superelliptic.poly))
print("Weights: {}".format(weights_non_superelliptic))
print("Is superelliptic? {}".format(is_superelliptic))
if not is_superelliptic:
    print("  Reason: {}".format(details))
EOF_PYTHON_SCRIPT

sage /tmp/temp_sage_script.py


In [ ]:
%%bash
source activate sage
export PYTHONPATH=$PYTHONPATH:/content

cat << 'EOF_PYTHON_SCRIPT' > /tmp/temp_sage_script.py
# SageMath/Python code to generate weighted projective superelliptic curves satisfying the theorem conditions
# Outputs: First curve that satisfies self-orthogonal C != C^perp, with Q parameters

from sage.all import GF, matrix, gcd, PolynomialRing, random, ZZ
from qwag.algeom.weighted_projective_space import WeightedProjectiveSpace
from qwag.algeom.weighted_curve import WeightedCurve, CustomLinearCode, construct_css_quantum_code

def generate_superelliptic_curve(q, weights, n_level, max_tries=1000):
    """
    Generate a weighted superelliptic curve y^n = f(x,z) over F_q in P(weights) satisfying theorem conditions.
    - q: prime power for F_q
    - weights: list [w0, w1, w2] for x,y,z (coprime, >=1)
    - n_level: superelliptic level n >=2
    - max_tries: try random f until conditions met

    Returns: (curve, G, C, Q_params) or (None, str) if no curve found
    """
    if gcd(weights) != 1:
        raise ValueError("Weights must be coprime")
    if len(weights) != 3:
        raise ValueError("Weights for P(w0,w1,w2)")

    F = GF(q)

    print(f"Starting search for curve with q={q}, weights={weights}, n_level={n_level}, max_tries={max_tries}")

    for try_num in range(max_tries):
        # Create an instance of our WeightedProjectiveSpace and its coord_ring first
        space = WeightedProjectiveSpace(F, weights)
        R = space.coord_ring # Use the polynomial ring from the space
        x, y, z = R.gens()   # Get variables from this specific ring (x0, x1, x2 typically)

        # Weighted degree for f = n * w1 (wt(y))
        deg_f = n_level * weights[1]

        # Generate a random polynomial f that is weighted homogeneous of degree deg_f in x and z
        f_terms = []
        k_x_gen = deg_f // weights[0]  # Highest x power

        for i in range(k_x_gen + 1):
            if (deg_f - i * weights[0]) % weights[2] == 0:
                j = (deg_f - i * weights[0]) // weights[2]
                # Ensure that coefficients are not all zero, especially for the leading term
                if i == k_x_gen and j == 0: # Monic highest degree term in x
                    f_terms.append(x**i * z**j)
                else:
                    # Random non-zero coefficient
                    coeff = F.random_element()
                    while coeff == F(0) and F.order() > 1: # ensure non-zero coeff if field is not trivial
                        coeff = F.random_element()
                    f_terms.append(coeff * x**i * z**j)
        f = sum(f_terms)


        # Full equation: y**n_level - f = 0
        poly = y**n_level - f

        try:
            curve = WeightedCurve(space, poly)
        except ValueError as e:
            print(f"DEBUG {try_num}: Curve construction failed: {e}. Skipping.")
            continue

        # Genus (using our implemented arithmetic genus)
        g = curve.arithmetic_genus()

        if g < 1:
            print(f"DEBUG {try_num}: Genus {g} < 1. Skipping.")
            continue

        # Rational points from our WeightedCurve method
        rational_points = curve.rational_points()
        n = len(rational_points)

        if n < 2*g:  # Need enough points for certain AG code bounds; this is a heuristic
            print(f"DEBUG {try_num}: {n} rational points < 2*g ({2*g}). Skipping.")
            continue

        # Using divisor degree for evaluation code (a common choice for some AG codes)
        divisor_degree_G = g - 1
        if divisor_degree_G < 0:
            print(f"DEBUG {try_num}: Divisor degree G = {divisor_degree_G} < 0. Skipping.")
            continue

        # Build the classical AG code C = C_L(D, G)
        C = curve.evaluation_code(rational_points, divisor_degree_G)

        if C.n == 0:
            print(f"DEBUG {try_num}: Code length is 0. Skipping.")
            continue

        # Check self-orthogonality
        if not C.is_self_orthogonal():
            print(f"DEBUG {try_num}: Code is NOT self-orthogonal. Skipping.")
            continue

        # Check not self-dual: k < n/2 (k is dimension of C_Z)
        # Here, we want C subset C^perp, but C != C^perp. For C=C_Z, C_X=C^perp
        C_dual = C.dual_code()
        if C.k == C_dual.k:
            print(f"DEBUG {try_num}: Code IS self-dual (k={C.k}, k_dual={C_dual.k}). Skipping.")
            continue

        # Construct quantum code Q via CSS
        quantum_params = construct_css_quantum_code(C, curve=curve)

        # Final check for non-trivial quantum code
        if quantum_params['k_Q'] > 0:
            # Found one!
            print(f"Found curve after {try_num+1} tries!")
            return curve, divisor_degree_G, [C.n, C.k, C.min_distance()], quantum_params
        else:
            print(f"DEBUG {try_num}: Quantum code is trivial (k_Q={quantum_params['k_Q']}). Skipping.")
            continue

    return None, "No curve found after max_tries"

# Example usage
q_val = 7
weights_val = [1,3,1]  # Coprime weights for x,y,z
n_level_val = 2  # Superelliptic level

result = generate_superelliptic_curve(q_val, weights_val, n_level_val, max_tries=5000) # Increased max_tries

if result[0] is not None: # Check if a curve was actually found
    curve, G_deg, C_params, Q_params = result # Unpack the tuple
    print("Found curve:", curve)
    print("G degree:", G_deg)
    print("Classical C:", C_params)
    print("Quantum Q:")
    print(f'  n_Q (code length): {Q_params["n_Q"]}')
    print(f'  k_Q (logical qubits): {Q_params["k_Q"]}')
    print(f'  d_Q (minimum distance): {Q_params["d_Q"]}')
    print(f'  d_Q (refined Singleton bound): {Q_params["d_Q_refined_bound"]}')
    print(f'  C_X generator matrix:\n{Q_params["C_X"].generator_matrix}')
    print(f'  C_Z generator matrix:\n{Q_params["C_Z"].generator_matrix}')
else:
    print(result[1]) # This will print the "No curve found..." message
EOF_PYTHON_SCRIPT

sage /tmp/temp_sage_script.py

## Final Task

### Subtask:
Provide a summary of the implemented quantum code construction function via CSS, its parameters, and any remaining limitations or next steps, particularly regarding the minimum distance calculation.


## Summary:

### Data Analysis Key Findings

*   A `min_distance` method was successfully integrated into the `CustomLinearCode` class. This method leverages SageMath's `minimum_distance()` function and appropriately returns 0 for codes with zero length ($n=0$) or zero dimension ($k=0$).
*   The `construct_css_quantum_code` function was updated to correctly calculate the quantum minimum distance ($d_Q$) by taking the minimum of the `min_distance()` of the $C_Z$ code and its dual, $C_X$.
*   For the provided test case, which involved a `WeightedCurve` defined by $poly = x*y$ over `GF(2)` with a `divisor_degree` of 0, the resulting classical code was confirmed to be self-orthogonal.
*   The `construct_css_quantum_code` function successfully determined the quantum code parameters for this test case as: $n_Q = 2$, $k_Q = 0$, and $d_Q = 2$.

### Insights or Next Steps

*   While the `min_distance` implementation relies on SageMath for accuracy and efficiency, exploring an internal, algorithmic implementation within the `CustomLinearCode` class could provide a deeper understanding and reduce external dependencies for specific use cases.
*   The current test case yielded $k_Q = 0$, representing a trivial quantum code. It would be beneficial to conduct further tests with curves and parameters that are expected to produce non-trivial quantum codes (i.e., $k_Q > 0$) to ensure the robustness and correctness of the `construct_css_quantum_code` function across a wider range of inputs.
